In [ ]:
%%capture
try:
    import ipyevents, pydicom, PIL, numpy, requests
except ImportError:
    %pip install -q ipyevents pydicom Pillow numpy requests

In [ ]:
"""Self-contained Inference dashboard — single-file build for XNAT.

Consolidated from utils/* in the AI-NB-POC repo so the notebook can be
uploaded to XNAT JupyterHub without a sibling utils/ package. Edits made in
the in-repo utils/ tree need to be re-synced into this cell.
"""

from __future__ import annotations

import io
import json
import os
import time
import warnings
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pydicom
import requests
import traitlets
from IPython.display import display
from PIL import Image
from ipyevents import Event
from pydicom.pixel_data_handlers.util import apply_modality_lut, apply_voi_lut

warnings.filterwarnings("ignore", category=UserWarning)

# === Embedded dark theme (utils/styles/dark.css) ===========================
_DARK_CSS_TEXT = r"""
/* ============================================================
   AI-NB-POC — modern-gui dark theme
   Single stylesheet loaded via <style> HTML widget at app boot.
   ============================================================ */

:root {
    --bg-root: #0f1419;
    --bg-panel: #1a1f2e;
    --bg-panel-alt: #232940;
    --bg-elevated: #2a3147;
    --bg-input: #1f2436;
    --border: #252b3d;
    --border-strong: #344058;
    --text: #e4e7eb;
    --text-muted: #7a8290;
    --text-dim: #9aa0ac;
    --accent: #1e88e5;
    --accent-hover: #2196f3;
    --accent-dim: rgba(30, 136, 229, 0.15);
    --severity-normal-bg: rgba(56, 142, 60, 0.18);
    --severity-normal-fg: #66bb6a;
    --severity-moderate-bg: rgba(245, 124, 0, 0.18);
    --severity-moderate-fg: #ffb74d;
    --severity-severe-bg: rgba(211, 47, 47, 0.20);
    --severity-severe-fg: #ef5350;
    --warn-bg: rgba(245, 124, 0, 0.12);
    --warn-border: rgba(245, 124, 0, 0.45);
    --warn-fg: #ffb74d;
    --status-ready: #4caf50;
    --status-running: #ffb74d;
    --status-error: #ef5350;
}

/* --- Voila / page background ----------------------------------------- */

body, .jp-Notebook, .jp-MainAreaWidget, #site, #main {
    background-color: var(--bg-root) !important;
    color: var(--text);
}

.jp-Cell, .jp-CodeCell, .jp-RenderedHTMLCommon {
    background-color: transparent !important;
    color: var(--text) !important;
}

/* --- App root --------------------------------------------------------- */

.nbpoc-app {
    background-color: var(--bg-root);
    color: var(--text);
    font-family: -apple-system, BlinkMacSystemFont, "Inter", "Segoe UI",
                 Roboto, "Helvetica Neue", Arial, sans-serif;
    font-size: 13px;
}

.nbpoc-app *, .nbpoc-app *::before, .nbpoc-app *::after {
    box-sizing: border-box;
}

/* AppLayout grid — keep panes flush against each other */
.nbpoc-app .jupyter-widgets.widget-gridbox,
.nbpoc-app .widget-appLayout {
    gap: 0 !important;
}

/* --- Top toolbar ------------------------------------------------------ */

.nbpoc-toolbar {
    display: flex;
    align-items: center;
    justify-content: space-between;
    height: 52px;
    padding: 0 14px;
    background-color: var(--bg-panel);
    border-bottom: 1px solid var(--border);
    color: var(--text);
    font-size: 13px;
    gap: 16px;
}

.nbpoc-toolbar-group {
    display: flex;
    align-items: center;
    gap: 6px;
    height: 100%;
}

.nbpoc-toolbar-brand {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    padding: 4px 10px;
    background-color: var(--bg-input);
    border: 1px solid var(--border);
    border-radius: 14px;
    color: var(--text-muted);
    font-size: 12px;
    cursor: default;
}

.nbpoc-toolbar-brand .dot {
    display: inline-block;
    width: 14px;
    height: 14px;
    border-radius: 3px;
    background: linear-gradient(135deg, var(--accent), #00bcd4);
}

.nbpoc-toolbar-btn {
    height: 32px;
    padding: 0 12px;
    background-color: transparent;
    border: 1px solid transparent;
    border-radius: 6px;
    color: var(--text-dim);
    font-size: 12.5px;
    font-weight: 500;
    cursor: pointer;
    transition: background-color 0.12s ease, color 0.12s ease;
    display: inline-flex;
    align-items: center;
    gap: 6px;
}

.nbpoc-toolbar-btn:hover {
    background-color: var(--bg-elevated);
    color: var(--text);
}

.nbpoc-toolbar-btn.active {
    background-color: var(--accent-dim);
    color: var(--accent);
    border-color: var(--accent);
}

.nbpoc-toolbar-icon {
    height: 32px;
    width: 32px;
    padding: 0;
    background-color: transparent;
    border: 1px solid transparent;
    border-radius: 6px;
    color: var(--text-dim);
    cursor: pointer;
    display: inline-flex;
    align-items: center;
    justify-content: center;
    font-size: 13px;
}

.nbpoc-toolbar-icon:hover {
    background-color: var(--bg-elevated);
    color: var(--text);
}

.nbpoc-toolbar-divider {
    width: 1px;
    height: 22px;
    background-color: var(--border-strong);
    margin: 0 4px;
}

.nbpoc-toolbar-fps {
    color: var(--text-muted);
    font-size: 12px;
    font-variant-numeric: tabular-nums;
    padding: 0 6px;
}

/* --- Left sidebar (scan list) ---------------------------------------- */

.nbpoc-sidebar {
    background-color: var(--bg-panel);
    border-right: 1px solid var(--border);
    height: 100%;
    overflow-y: auto;
    padding: 0;
}

.nbpoc-breadcrumb {
    padding: 10px 14px;
    font-size: 11.5px;
    color: var(--text-muted);
    border-bottom: 1px solid var(--border);
}

.nbpoc-breadcrumb .sep {
    color: var(--text-dim);
    margin: 0 5px;
}

.nbpoc-breadcrumb .crumb {
    color: var(--text-dim);
}

.nbpoc-breadcrumb .crumb.current {
    color: var(--text);
}

.nbpoc-filter-row {
    padding: 8px 12px;
    border-bottom: 1px solid var(--border);
    background-color: var(--bg-panel);
}

.nbpoc-scan-list {
    padding: 6px 0 12px;
}

.nbpoc-scan-row {
    display: flex;
    align-items: center;
    gap: 10px;
    padding: 9px 14px;
    cursor: pointer;
    border-left: 3px solid transparent;
    transition: background-color 0.1s ease;
}

.nbpoc-scan-row:hover {
    background-color: var(--bg-panel-alt);
}

.nbpoc-scan-row.selected {
    background-color: var(--bg-panel-alt);
    border-left-color: var(--accent);
}

.nbpoc-scan-icon {
    width: 28px;
    height: 28px;
    display: inline-flex;
    align-items: center;
    justify-content: center;
    background-color: var(--bg-elevated);
    border-radius: 4px;
    color: #ffb74d;
    font-size: 14px;
    flex-shrink: 0;
}

.nbpoc-scan-meta {
    flex: 1;
    min-width: 0;
}

.nbpoc-scan-title {
    display: flex;
    align-items: center;
    gap: 6px;
    font-weight: 600;
    color: var(--text);
    font-size: 12.5px;
    line-height: 1.3;
}

.nbpoc-scan-title .idx {
    color: var(--text-dim);
    font-weight: 500;
    margin-right: 2px;
}

.nbpoc-chip {
    display: inline-block;
    padding: 1px 6px;
    background-color: var(--bg-elevated);
    border: 1px solid var(--border-strong);
    border-radius: 3px;
    color: var(--text-muted);
    font-size: 9.5px;
    font-weight: 700;
    letter-spacing: 0.4px;
    text-transform: uppercase;
}

.nbpoc-scan-sub {
    color: var(--text-muted);
    font-size: 11px;
    line-height: 1.4;
    margin-top: 1px;
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
}

/* --- Center viewer panel --------------------------------------------- */

.nbpoc-viewer {
    background-color: var(--bg-root);
    height: 100%;
    width: 100%;
    position: relative;
    display: flex;
    flex-direction: column;
}

.nbpoc-viewer-canvas {
    flex: 1;
    position: relative;
    display: flex;
    align-items: center;
    justify-content: center;
    background-color: #000;
    overflow: hidden;
    min-height: 500px;
}

.nbpoc-viewer-canvas .widget-image img,
.nbpoc-viewer-canvas .widget-image {
    max-width: 100%;
    max-height: 100%;
    object-fit: contain;
    background-color: transparent;
}

.nbpoc-viewer-overlay {
    position: absolute;
    color: var(--text);
    font-size: 11.5px;
    line-height: 1.45;
    text-shadow: 0 1px 2px rgba(0, 0, 0, 0.7);
    font-variant-numeric: tabular-nums;
    pointer-events: none;
    z-index: 5;
}

.nbpoc-viewer-overlay.tl { top: 12px; left: 16px; }
.nbpoc-viewer-overlay.tr { top: 12px; right: 16px; text-align: right; }
.nbpoc-viewer-overlay.bl { bottom: 36px; left: 16px; }
.nbpoc-viewer-overlay.br { bottom: 36px; right: 16px; text-align: right; }

.nbpoc-viewer-overlay .title {
    font-weight: 600;
    font-size: 12.5px;
}

.nbpoc-viewer-overlay .muted {
    color: var(--text-dim);
}

.nbpoc-viewer-scrub {
    height: 28px;
    background-color: var(--bg-panel);
    border-top: 1px solid var(--border);
    padding: 0 12px;
    display: flex;
    align-items: center;
}

/* Vertical slice scrubber pinned to the right edge of the canvas */
.nbpoc-viewer-slider {
    position: absolute;
    right: 14px;
    top: 50%;
    transform: translateY(-50%);
    z-index: 6;
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 6px;
    padding: 10px 6px;
    background-color: rgba(26, 31, 46, 0.85);
    border: 1px solid var(--border);
    border-radius: 8px;
    box-shadow: 0 2px 6px rgba(0, 0, 0, 0.35);
    backdrop-filter: blur(4px);
}

.nbpoc-viewer-slider .widget-vslider,
.nbpoc-viewer-slider .widget-slider {
    margin: 0;
}

/* --- Right inference panel -------------------------------------------- */

.nbpoc-inference {
    background-color: var(--bg-panel);
    border-left: 1px solid var(--border);
    height: 100%;
    overflow-y: auto;
    color: var(--text);
}

.nbpoc-inference-header {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 12px 14px;
    border-bottom: 1px solid var(--border);
}

.nbpoc-inference-header .title {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    font-size: 13.5px;
    font-weight: 600;
}

.nbpoc-inference-header .title .glyph {
    color: var(--accent);
}

.nbpoc-inference-body {
    padding: 12px 14px;
    display: flex;
    flex-direction: column;
    gap: 12px;
}

.nbpoc-disclaimer {
    background-color: var(--warn-bg);
    border: 1px solid var(--warn-border);
    border-radius: 6px;
    padding: 10px 12px;
    color: var(--warn-fg);
    font-size: 11.5px;
    line-height: 1.4;
}

.nbpoc-status-row {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 4px 0;
}

.nbpoc-status {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    font-size: 12.5px;
}

.nbpoc-status .dot {
    width: 8px;
    height: 8px;
    border-radius: 50%;
    background-color: var(--status-ready);
    box-shadow: 0 0 0 3px rgba(76, 175, 80, 0.18);
}

.nbpoc-status.running .dot { background-color: var(--status-running);
    box-shadow: 0 0 0 3px rgba(255, 183, 77, 0.18); }
.nbpoc-status.error .dot { background-color: var(--status-error);
    box-shadow: 0 0 0 3px rgba(239, 83, 80, 0.18); }

.nbpoc-stop-btn {
    height: 26px;
    padding: 0 12px;
    background-color: transparent;
    border: 1px solid var(--border-strong);
    color: var(--text-dim);
    border-radius: 4px;
    font-size: 11.5px;
    cursor: pointer;
}

.nbpoc-stop-btn:hover { color: var(--text); }

.nbpoc-analyze-btn,
.nbpoc-inference button.nbpoc-analyze-btn,
.nbpoc-inference .widget-button.nbpoc-analyze-wrap button {
    background-color: var(--accent) !important;
    color: #fff !important;
    font-weight: 600 !important;
    height: 38px !important;
    width: 100% !important;
    border: none !important;
    border-radius: 6px !important;
    cursor: pointer;
    font-size: 13px !important;
    box-shadow: 0 1px 2px rgba(0, 0, 0, 0.3);
    transition: background-color 0.12s ease;
}

.nbpoc-analyze-btn:hover,
.nbpoc-inference .widget-button.nbpoc-analyze-wrap button:hover {
    background-color: var(--accent-hover) !important;
}

.nbpoc-section-label {
    color: var(--text-muted);
    font-size: 10.5px;
    font-weight: 700;
    letter-spacing: 0.6px;
    padding: 6px 0 2px;
    text-transform: uppercase;
}

/* --- Result cards ----------------------------------------------------- */

.nbpoc-card {
    background-color: var(--bg-panel-alt);
    border: 1px solid var(--border-strong);
    border-radius: 6px;
    padding: 10px 12px;
    color: var(--text);
    font-size: 12.5px;
    line-height: 1.4;
}

.nbpoc-card.normal { border-left: 3px solid var(--severity-normal-fg); }
.nbpoc-card.moderate { border-left: 3px solid var(--severity-moderate-fg); }
.nbpoc-card.severe { border-left: 3px solid var(--severity-severe-fg); }

.nbpoc-card-head {
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    gap: 8px;
    margin-bottom: 4px;
}

.nbpoc-card-title {
    font-weight: 500;
    color: var(--text);
    flex: 1;
    line-height: 1.35;
}

.nbpoc-card-severity {
    flex-shrink: 0;
    padding: 2px 7px;
    border-radius: 3px;
    font-size: 10px;
    font-weight: 700;
    letter-spacing: 0.4px;
    text-transform: lowercase;
}

.nbpoc-card-severity.normal { background-color: var(--severity-normal-bg); color: var(--severity-normal-fg); }
.nbpoc-card-severity.moderate { background-color: var(--severity-moderate-bg); color: var(--severity-moderate-fg); }
.nbpoc-card-severity.severe { background-color: var(--severity-severe-bg); color: var(--severity-severe-fg); }

.nbpoc-card-meta {
    color: var(--text-muted);
    font-size: 11px;
    margin: 4px 0 8px;
    display: flex;
    flex-wrap: wrap;
    gap: 6px;
    align-items: center;
}

.nbpoc-card-meta a {
    color: var(--accent);
    text-decoration: none;
}

.nbpoc-card-meta .sep { color: var(--text-dim); }

.nbpoc-card-actions {
    display: flex;
    gap: 6px;
    margin-top: 6px;
    align-items: center;
    flex-wrap: wrap;
}

.nbpoc-card-actions .widget-checkbox {
    margin: 0 6px 0 0;
    padding: 0;
    flex-shrink: 0;
}

.nbpoc-card-actions .widget-checkbox label {
    font-size: 11.5px;
    color: var(--text-dim);
    display: inline-flex;
    align-items: center;
    gap: 4px;
}

.nbpoc-card-action {
    height: 26px;
    padding: 0 10px;
    background-color: var(--bg-elevated);
    border: 1px solid var(--border-strong);
    color: var(--text-dim);
    border-radius: 4px;
    font-size: 11.5px;
    cursor: pointer;
    display: inline-flex;
    align-items: center;
    gap: 4px;
}

.nbpoc-card-action:hover { color: var(--text); background-color: var(--bg-panel); }
.nbpoc-card-action.accept:hover { color: var(--severity-normal-fg); border-color: var(--severity-normal-fg); }
.nbpoc-card-action.reject:hover { color: var(--severity-severe-fg); border-color: var(--severity-severe-fg); }
.nbpoc-card-action.edit:hover { color: var(--accent); border-color: var(--accent); }

.nbpoc-card-drawer {
    margin-top: 10px;
    padding-top: 10px;
    border-top: 1px dashed var(--border-strong);
}

.nbpoc-card-drawer.hidden { display: none; }

.nbpoc-card-diag-chip {
    margin-top: 8px;
    font-size: 11px;
    color: var(--text-dim);
    cursor: pointer;
}

.nbpoc-card-diag-chip:hover { color: var(--accent); }

/* --- Footer ----------------------------------------------------------- */

.nbpoc-footer {
    height: 28px;
    background-color: var(--bg-panel);
    border-top: 1px solid var(--border);
    display: flex;
    align-items: center;
    justify-content: flex-end;
    padding: 0 14px;
    color: var(--text-muted);
    font-size: 11px;
}

/* --- ipywidgets overrides for dark mode ------------------------------- */

.nbpoc-app .widget-button button,
.nbpoc-app button.jupyter-button {
    background-color: var(--bg-elevated);
    color: var(--text);
    border: 1px solid var(--border-strong);
    border-radius: 4px;
    font-size: 12px;
    height: 28px;
    box-shadow: none;
    transition: background-color 0.12s ease;
}

.nbpoc-app .widget-button button:hover,
.nbpoc-app button.jupyter-button:hover {
    background-color: var(--bg-panel-alt);
    color: var(--text);
}

.nbpoc-app .widget-button button.mod-primary {
    background-color: var(--accent);
    color: #fff;
    border-color: var(--accent);
}
.nbpoc-app .widget-button button.mod-primary:hover {
    background-color: var(--accent-hover);
}

.nbpoc-app .widget-button button.mod-warning {
    background-color: rgba(245, 124, 0, 0.20);
    color: var(--warn-fg);
    border-color: var(--warn-border);
}

.nbpoc-app .widget-button button.mod-info {
    background-color: var(--accent-dim);
    color: var(--accent);
    border-color: var(--accent);
}

.nbpoc-app .widget-text input,
.nbpoc-app .widget-textarea textarea,
.nbpoc-app .widget-password input,
.nbpoc-app .widget-numeric input,
.nbpoc-app .widget-dropdown select,
.nbpoc-app .widget-combobox input,
.nbpoc-app input[type="text"],
.nbpoc-app input[type="number"],
.nbpoc-app select {
    background-color: var(--bg-input) !important;
    color: var(--text) !important;
    border: 1px solid var(--border-strong) !important;
    border-radius: 4px !important;
    font-size: 12px !important;
    box-shadow: none !important;
}

.nbpoc-app .widget-text input:focus,
.nbpoc-app .widget-textarea textarea:focus,
.nbpoc-app .widget-numeric input:focus,
.nbpoc-app .widget-dropdown select:focus,
.nbpoc-app input:focus,
.nbpoc-app select:focus {
    border-color: var(--accent) !important;
    box-shadow: 0 0 0 2px rgba(30, 136, 229, 0.18) !important;
    outline: none !important;
}

.nbpoc-app label,
.nbpoc-app .widget-label,
.nbpoc-app .jupyter-widgets {
    color: var(--text);
}

.nbpoc-app .widget-checkbox label {
    color: var(--text);
    font-size: 12px;
}

.nbpoc-app .widget-checkbox input[type="checkbox"] {
    accent-color: var(--accent);
}

/* IntSlider track in dark mode */
.nbpoc-app .widget-slider .slider-track,
.nbpoc-app .ui-slider {
    background-color: var(--bg-input) !important;
    border-color: var(--border-strong) !important;
}

.nbpoc-app .ui-slider .ui-slider-handle {
    background-color: var(--accent) !important;
    border-color: var(--accent) !important;
}

.nbpoc-app .ui-slider .ui-slider-range,
.nbpoc-app .noUi-connect {
    background-color: var(--accent) !important;
}

/* Generic select widget (file_browser uses widgets.Select) */
.nbpoc-app .widget-select select option {
    background-color: var(--bg-input);
    color: var(--text);
}

/* Spinner used by segmentation_tab — keep but recolor */
.nbpoc-app .nbpoc-spinner {
    width: 14px;
    height: 14px;
    border: 2px solid var(--border-strong);
    border-top-color: var(--accent);
    border-radius: 50%;
    animation: nbpoc-spin 0.8s linear infinite;
    display: inline-block;
    vertical-align: middle;
}

@keyframes nbpoc-spin { to { transform: rotate(360deg); } }

/* Hide voila/jupyter chrome we don't want */
.nbpoc-app .jp-OutputArea-prompt,
.nbpoc-app .jp-InputArea-prompt {
    display: none !important;
}

"""

# === Config (utils/config.py) ==========================================

"""Inference endpoint and path-translation configuration for the Medical AI POC."""

import os

# Inference endpoint template. {model} is substituted with the selected model
# name, e.g. duneai-nsclc ->
# http://duneai-nsclc-xnat.tap.dev.embarklabs.ai/v1/models/duneai-nsclc:predict
INFERENCE_URL_TEMPLATE = os.environ.get(
    "INFERENCE_URL_TEMPLATE",
    "http://{model}-xnat.tap.dev.embarklabs.ai/v1/models/{model}:predict",
)

# KServe scale-to-zero cold starts can take several minutes while the model
# weights mount and vLLM spins up.
REQUEST_TIMEOUT = int(os.environ.get("REQUEST_TIMEOUT", "900"))

# PVC path translation. The JupyterHub kernel and the inference pod mount the
# same archive under different prefixes, and the first segment under AI-POC/
# differs between the two views (notebook = "experiments", XNAT archive =
# "arc001"). SEGMENT_MAP rewrites that first segment; anything not in the map
# passes through unchanged.
LOCAL_DATA_ROOT = os.environ.get("LOCAL_DATA_ROOT", "/data/projects/AI-POC")
POD_DATA_ROOT = os.environ.get("POD_DATA_ROOT", "/data/xnat/archive/AI-POC")
SEGMENT_MAP = {"experiments": "arc001"}

# === DICOM utilities (utils/dicom_utils.py) ============================

"""DICOM reading, windowing, and conversion utilities."""


import io

import numpy as np
import pydicom
from pydicom.pixel_data_handlers.util import apply_modality_lut, apply_voi_lut
from PIL import Image


_KNOWN_NON_DICOM = {
    ".txt", ".csv", ".json", ".xml", ".html", ".log", ".md",
    ".yaml", ".yml", ".cfg", ".ini", ".conf",
    ".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff", ".tif", ".svg",
    ".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
    ".py", ".js", ".ts", ".java", ".c", ".cpp", ".h", ".rs", ".go",
    ".zip", ".tar", ".gz", ".bz2", ".rar", ".7z", ".xz",
    ".exe", ".dll", ".so", ".dylib", ".bin",
    ".mp4", ".avi", ".mov", ".mkv", ".mp3", ".wav", ".flac",
    ".nii", ".nrrd", ".mha", ".mhd", ".stl", ".obj", ".vtk",
    ".sh", ".bat", ".ps1",
}


def is_dicom_candidate(p):
    """Check if a path is likely a DICOM file.

    Uses an exclusion-based approach: any file that doesn't have a known
    non-DICOM extension is treated as a candidate.  This catches numeric
    extensions (.001, .1), Siemens .ima, UID-fragment names, and
    extension-less files common in PACS exports.
    """
    if p.is_dir():
        return False
    return p.suffix.lower() not in _KNOWN_NON_DICOM


def is_nifti_file(p):
    """Check if a path is a NIfTI file (.nii or .nii.gz)."""
    if p.is_dir():
        return False
    name = p.name.lower()
    return name.endswith(".nii") or name.endswith(".nii.gz")


def read_dicom(path):
    """Read a DICOM file, return Dataset or None on failure."""
    try:
        return pydicom.dcmread(str(path), force=True)
    except Exception:
        return None


def apply_windowing(ds):
    """Apply modality + VOI LUTs via pydicom, return float64 array in [0, 1]."""
    pixels = ds.pixel_array
    pixels = apply_modality_lut(pixels, ds)
    pixels = apply_voi_lut(pixels, ds)
    pixels = pixels.astype(np.float64)
    lo, hi = float(pixels.min()), float(pixels.max())
    if hi - lo < 1e-6:
        return np.zeros_like(pixels, dtype=np.float64)
    return (pixels - lo) / (hi - lo)


def dicom_to_pil(ds):
    """Convert DICOM Dataset to an 8-bit grayscale PIL Image."""
    arr = apply_windowing(ds)
    # Invert MONOCHROME1 (white = low values)
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        arr = 1.0 - arr
    return Image.fromarray((arr * 255).astype(np.uint8), mode="L")


def dicom_to_png_bytes(ds):
    """Convert DICOM to PNG bytes for the endpoint."""
    buf = io.BytesIO()
    dicom_to_pil(ds).save(buf, format="PNG")
    return buf.getvalue()


def load_series(directory_path):
    """Load all valid DICOM files in a directory as a sorted series.

    Returns list of (dataset, png_bytes) tuples sorted by InstanceNumber,
    SliceLocation, or filename as fallback.  Returns [] if <1 valid DICOM.
    """
    from pathlib import Path

    directory_path = Path(directory_path)
    candidates = [p for p in directory_path.iterdir() if is_dicom_candidate(p)]

    loaded = []
    for p in candidates:
        ds = read_dicom(p)
        if ds is None:
            continue
        try:
            _ = ds.pixel_array
            png = dicom_to_png_bytes(ds)
        except Exception:
            continue
        loaded.append((ds, png, p.name))

    def _sort_key(item):
        ds, _png, fname = item
        inst = getattr(ds, "InstanceNumber", None)
        if inst is not None:
            try:
                return (0, int(inst), fname)
            except (TypeError, ValueError):
                pass
        sloc = getattr(ds, "SliceLocation", None)
        if sloc is not None:
            try:
                return (1, float(sloc), fname)
            except (TypeError, ValueError):
                pass
        return (2, 0, fname)

    loaded.sort(key=_sort_key)
    return [(ds, png) for ds, png, _fname in loaded]


_DEFAULT_SEG_PALETTE = [
    (239, 83, 80),   # red
    (66, 165, 245),  # blue
    (102, 187, 106), # green
    (255, 167, 38),  # orange
    (171, 71, 188),  # purple
    (38, 198, 218),  # cyan
    (255, 238, 88),  # yellow
    (141, 110, 99),  # brown
    (236, 64, 122),  # pink
    (156, 204, 101), # lime
]


def _cielab_dicom_to_rgb(lab):
    """DICOM RecommendedDisplayCIELabValue (uint16 x3) -> sRGB 0-255 tuple.

    DICOM encodes L in [0, 65535] -> [0, 100]; a,b in [0, 65535] -> [-128, 127].
    """
    L = float(lab[0]) / 65535.0 * 100.0
    a = float(lab[1]) / 65535.0 * 255.0 - 128.0
    b = float(lab[2]) / 65535.0 * 255.0 - 128.0

    fy = (L + 16.0) / 116.0
    fx = a / 500.0 + fy
    fz = fy - b / 200.0

    def _f_inv(t):
        t3 = t ** 3
        return t3 if t3 > 0.008856 else (t - 16.0 / 116.0) / 7.787

    # D65 reference white
    X = 0.95047 * _f_inv(fx)
    Y = 1.00000 * _f_inv(fy)
    Z = 1.08883 * _f_inv(fz)

    r =  3.2406 * X - 1.5372 * Y - 0.4986 * Z
    g = -0.9689 * X + 1.8758 * Y + 0.0415 * Z
    bl = 0.0557 * X - 0.2040 * Y + 1.0570 * Z

    def _gamma(u):
        u = max(0.0, u)
        return 1.055 * (u ** (1 / 2.4)) - 0.055 if u > 0.0031308 else 12.92 * u

    return tuple(
        max(0, min(255, int(round(_gamma(c) * 255))))
        for c in (r, g, bl)
    )


def _segment_color(seg, num):
    lab = getattr(seg, "RecommendedDisplayCIELabValue", None)
    if lab is not None and len(lab) == 3:
        try:
            return _cielab_dicom_to_rgb(lab)
        except Exception:
            pass
    return _DEFAULT_SEG_PALETTE[(num - 1) % len(_DEFAULT_SEG_PALETTE)]


def _source_sop_uid(fg):
    try:
        drv = fg.DerivationImageSequence[0]
        src = drv.SourceImageSequence[0]
        return str(src.ReferencedSOPInstanceUID)
    except (AttributeError, IndexError):
        return None


def _segment_number(fg):
    try:
        seg_id = fg.SegmentIdentificationSequence[0]
        return int(seg_id.ReferencedSegmentNumber)
    except (AttributeError, IndexError, ValueError):
        return None


def load_dicom_seg(path):
    """Load a DICOM SEG file into per-slice masks keyed by source SOP UID.

    Returns dict with:
        by_source_sop: {sop_instance_uid: {segment_number: bool ndarray (H, W)}}
        segments:      {segment_number: {"label": str, "color": (r, g, b)}}
        referenced_series_uid: SeriesInstanceUID of the source series (or None)
    """
    ds = pydicom.dcmread(str(path))
    if getattr(ds, "Modality", "") != "SEG":
        raise ValueError(
            f"Not a DICOM SEG file (Modality={getattr(ds, 'Modality', None)!r})"
        )

    segments = {}
    for seg in getattr(ds, "SegmentSequence", []):
        num = int(seg.SegmentNumber)
        segments[num] = {
            "label": getattr(seg, "SegmentLabel", f"Segment {num}"),
            "color": _segment_color(seg, num),
        }

    ref_series_uid = None
    rss = getattr(ds, "ReferencedSeriesSequence", None)
    if rss and len(rss) > 0:
        ref_series_uid = getattr(rss[0], "SeriesInstanceUID", None)

    frames = ds.pixel_array
    if frames.ndim == 2:
        frames = frames[None, ...]

    by_source_sop = {}
    for i, fg in enumerate(ds.PerFrameFunctionalGroupsSequence):
        sop_uid = _source_sop_uid(fg)
        seg_num = _segment_number(fg)
        if sop_uid is None or seg_num is None:
            continue
        mask = frames[i].astype(bool)
        if not mask.any():
            continue
        by_source_sop.setdefault(sop_uid, {})[seg_num] = mask

    return {
        "by_source_sop": by_source_sop,
        "segments": segments,
        "referenced_series_uid": ref_series_uid,
    }


def composite_overlay(base_pil, segment_masks, segments, alpha=0.4):
    """Blend colored segmentation masks onto a base image.

    base_pil:       PIL.Image (any mode; converted to RGB internally)
    segment_masks:  {segment_number: bool ndarray (H, W)} for this slice; may be empty
    segments:       {segment_number: {"label": str, "color": (r, g, b)}}
    alpha:          overlay opacity, 0..1

    Masks whose shape differs from the base are resampled with
    nearest-neighbor so SEGs produced at a resampled resolution
    (e.g. TotalSegmentator --fast) still render correctly.

    Returns an RGB PIL.Image.
    """
    base_rgb = np.array(base_pil.convert("RGB"), dtype=np.float32)
    h, w = base_rgb.shape[:2]

    if not segment_masks:
        return Image.fromarray(base_rgb.astype(np.uint8), mode="RGB")

    for seg_num, mask in segment_masks.items():
        if mask.shape != (h, w):
            mask_pil = Image.fromarray(
                (mask.astype(np.uint8) * 255), mode="L"
            ).resize((w, h), Image.NEAREST)
            mask = np.array(mask_pil) > 127
        color = np.array(segments[seg_num]["color"], dtype=np.float32)
        base_rgb[mask] = (1.0 - alpha) * base_rgb[mask] + alpha * color

    return Image.fromarray(np.clip(base_rgb, 0, 255).astype(np.uint8), mode="RGB")


def extract_metadata(ds):
    """Pull a curated set of DICOM tags for display."""
    tags = [
        ("Patient", "PatientName"), ("Patient ID", "PatientID"),
        ("Study Date", "StudyDate"), ("Study", "StudyDescription"),
        ("Series", "SeriesDescription"), ("Modality", "Modality"),
        ("Body Part", "BodyPartExamined"), ("Institution", "InstitutionName"),
        ("Size", None), ("Bits", "BitsAllocated"),
        ("Photometric", "PhotometricInterpretation"),
        ("Window Center", "WindowCenter"), ("Window Width", "WindowWidth"),
    ]
    rows = []
    for label, attr in tags:
        if label == "Size":
            r = getattr(ds, "Rows", "?")
            c = getattr(ds, "Columns", "?")
            rows.append((label, f"{r} x {c}"))
        else:
            val = getattr(ds, attr, None)
            if val is not None:
                rows.append((label, str(val)))
    return rows

# === HTTP client (utils/http_client.py) ================================

"""KServe predictor HTTP client and PVC path translation."""


from pathlib import Path

import requests



def translate_path(local_path: str | Path) -> str:
    """Map a kernel-visible path under LOCAL_DATA_ROOT to the pod's mount.

    Rewrites the first path segment under LOCAL_DATA_ROOT using SEGMENT_MAP
    (e.g. 'experiments' -> 'arc001'); segments not in the map pass through.
    """
    p = Path(local_path).resolve()
    local_root = Path(LOCAL_DATA_ROOT).resolve()
    try:
        rel = p.relative_to(local_root)
    except ValueError as e:
        raise ValueError(
            f"Path {p} is outside {local_root}; cannot translate to pod path."
        ) from e
    parts = list(rel.parts)
    if parts and parts[0] in SEGMENT_MAP:
        parts[0] = SEGMENT_MAP[parts[0]]
    return str(Path(POD_DATA_ROOT).joinpath(*parts)) if parts else POD_DATA_ROOT


def translate_pod_path_to_local(pod_path: str | Path) -> str:
    """Map a pod-side path under POD_DATA_ROOT back to the kernel's local view.

    Inverse of translate_path: rewrites the first path segment using the
    reverse SEGMENT_MAP ('arc001' -> 'experiments') and swaps POD_DATA_ROOT
    for LOCAL_DATA_ROOT. Used to resolve paths returned by the totalseg
    predictor (e.g. the seg_path in a response) so the notebook kernel can
    open them.
    """
    p = Path(pod_path)
    pod_root = Path(POD_DATA_ROOT)
    try:
        rel = p.relative_to(pod_root)
    except ValueError as e:
        raise ValueError(
            f"Path {p} is outside {pod_root}; cannot translate to local path."
        ) from e
    reverse_map = {v: k for k, v in SEGMENT_MAP.items()}
    parts = list(rel.parts)
    if parts and parts[0] in reverse_map:
        parts[0] = reverse_map[parts[0]]
    return str(Path(LOCAL_DATA_ROOT).joinpath(*parts)) if parts else LOCAL_DATA_ROOT


def predict(payload: dict, model_name: str) -> dict:
    """POST a payload to the KServe predictor for the given model."""
    url = INFERENCE_URL_TEMPLATE.format(model=model_name)
    resp = requests.post(
        url,
        json=payload,
        headers={"Content-Type": "application/json"},
        timeout=REQUEST_TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()

# === App state (utils/state.py) ========================================

"""Centralized application state for the MedGemma dashboard."""


import traitlets


class AppState(traitlets.HasTraits):
    """Single source of truth shared across all dashboard components."""

    # Currently selected single DICOM (viewer + payload source)
    current_ds = traitlets.Any(default_value=None)
    current_png_bytes = traitlets.Bytes(default_value=b"")
    current_file_name = traitlets.Unicode(default_value="")
    current_file_path = traitlets.Unicode(default_value="")

    # DICOM series state
    series_datasets = traitlets.List(default_value=[])
    series_png_cache = traitlets.List(default_value=[])
    series_index = traitlets.Int(default_value=0)
    series_dir_name = traitlets.Unicode(default_value="")
    series_dir_path = traitlets.Unicode(default_value="")

# === Top toolbar (utils/components/top_toolbar.py) =====================

"""Top toolbar — visual stub matching the clinical-workstation reference.

Every button is a visual no-op for the modern-gui refresh; the real controls
(task dropdown, run button, masks/metadata visibility) live in the inference
panel and elsewhere. This module just renders the chrome.
"""


import ipywidgets as widgets


def _toolbar_btn(label: str, *, active: bool = False, dropdown: bool = False) -> str:
    cls = "nbpoc-toolbar-btn" + (" active" if active else "")
    arrow = " <span style='font-size:9px;opacity:0.7;'>&#x25BE;</span>" if dropdown else ""
    return f"<button class='{cls}' type='button'>{label}{arrow}</button>"


def _toolbar_icon(glyph: str, title: str = "") -> str:
    return (
        f"<button class='nbpoc-toolbar-icon' type='button' title='{title}'>"
        f"{glyph}</button>"
    )


def build_top_toolbar() -> widgets.HTML:
    """Return the top toolbar as a single HTML widget."""

    left = (
        "<div class='nbpoc-toolbar-group'>"
        "<span class='nbpoc-toolbar-brand'>"
        "<span class='dot'></span>"
        "<span>cnda.wustl.edu</span>"
        "</span>"
        + _toolbar_icon("&#x25A0;", "Single view")
        + _toolbar_icon("&#x25AE;&#x25AE;", "1&times;2")
        + _toolbar_icon("&#x25A6;", "2&times;2")
        + "<span class='nbpoc-toolbar-divider'></span>"
        + _toolbar_btn("Protocol", dropdown=True)
        + "</div>"
    )

    center = (
        "<div class='nbpoc-toolbar-group'>"
        + _toolbar_btn("MPR")
        + _toolbar_btn("W/L", active=True)
        + _toolbar_btn("Pan")
        + _toolbar_btn("Zoom")
        + _toolbar_btn("Measure", dropdown=True)
        + _toolbar_btn("Segment", dropdown=True)
        + _toolbar_btn("Presets", dropdown=True)
        + "</div>"
    )

    right = (
        "<div class='nbpoc-toolbar-group'>"
        + _toolbar_icon("&#x21BA;", "Undo")
        + _toolbar_icon("&#x21BB;", "Redo")
        + "<span class='nbpoc-toolbar-divider'></span>"
        + _toolbar_icon("&#x25C1;", "Previous")
        + _toolbar_icon("&#x25B7;", "Play")
        + _toolbar_icon("&#x25B7;&#x25B7;", "Next")
        + "<span class='nbpoc-toolbar-fps'>15 fps</span>"
        + "<span class='nbpoc-toolbar-divider'></span>"
        + _toolbar_icon("&#x22EE;", "More")
        + _toolbar_icon("&#x263C;", "Theme")
        + "</div>"
    )

    html = (
        f"<div class='nbpoc-toolbar'>{left}{center}{right}</div>"
    )
    return widgets.HTML(value=html)

# === Diagnostics panel (utils/components/diagnostics_panel.py) =========

"""Auto-discovering pipeline-diagnostics panel for the loaded DICOM series.

Mirrors ``seg_viewer``'s discovery: on series load, scans
``{state.series_dir_path}/../masks/*_diagnostics.json`` for sidecar timing
files emitted by transformer pods (BrainSeg today, DuneAI later). The
contract is generic across models — any JSON shaped as

    {model, version, schema_version, timings_ms, preprocess_breakdown?}

renders into a collapsible "Pipeline diagnostics" panel. Unknown
``schema_version`` values fall back to a raw-JSON dump.
"""


import json
from pathlib import Path

import ipywidgets as widgets

_DIAG_MASKS_SUBDIR = "masks"
_DIAGNOSTICS_SUFFIX = "_diagnostics.json"
_KNOWN_SCHEMA_VERSIONS = {1}

_TOP_STAGE_ORDER = ("preprocess", "model_inference", "postprocess", "total")


def _diag_muted_card(msg):
    return (
        f"<div style='color:var(--text-muted);background:var(--bg-panel-alt);"
        f"padding:8px 10px;border-left:3px solid var(--border-strong);"
        f"border-radius:4px;font-size:11.5px;'>{msg}</div>"
    )


def _diag_error_card(msg):
    return (
        f"<div style='color:var(--severity-severe-fg);"
        f"background:rgba(211,47,47,0.10);padding:8px 10px;"
        f"border-left:3px solid var(--severity-severe-fg);"
        f"border-radius:4px;font-size:11.5px;'>{msg}</div>"
    )


def _format_ms(value_ms):
    try:
        ms = float(value_ms)
    except (TypeError, ValueError):
        return str(value_ms)
    if ms >= 1000:
        return f"{ms / 1000:.2f} s"
    return f"{ms:.0f} ms"


def _stage_bar(label, value_ms, max_ms, accent="var(--accent)"):
    width_pct = 0
    if max_ms > 0:
        try:
            width_pct = max(0.0, min(100.0, float(value_ms) / float(max_ms) * 100.0))
        except (TypeError, ValueError):
            width_pct = 0
    return (
        f"<div style='display:flex;align-items:center;gap:10px;"
        f"padding:3px 0;font-size:11.5px;'>"
        f"<div style='flex:0 0 130px;color:var(--text-muted);'>{label}</div>"
        f"<div style='flex:1;background:var(--bg-input);border-radius:3px;"
        f"height:8px;position:relative;overflow:hidden;'>"
        f"<div style='width:{width_pct:.1f}%;background:{accent};"
        f"height:100%;border-radius:3px;'></div></div>"
        f"<div style='flex:0 0 70px;text-align:right;color:var(--text-dim);"
        f"font-family:monospace;'>{_format_ms(value_ms)}</div></div>"
    )


def _render_known_schema(data):
    timings = data.get("timings_ms") or {}
    breakdown = data.get("preprocess_breakdown") or {}
    model = data.get("model", "?")
    version = data.get("version", "?")

    top_values = [
        (key, timings[key]) for key in _TOP_STAGE_ORDER if key in timings
    ]
    bar_basis = max(
        (v for k, v in top_values if k != "total" and isinstance(v, (int, float))),
        default=0,
    )

    header = (
        f"<div style='font-size:10.5px;color:var(--text-muted);padding:0 0 6px;'>"
        f"<b style='color:var(--text);'>{model}</b> &middot; v{version}</div>"
    )
    rows = []
    for key, val in top_values:
        accent = "var(--text-dim)" if key == "total" else "var(--accent)"
        basis = max(bar_basis, val) if key == "total" else bar_basis
        rows.append(_stage_bar(key, val, basis, accent=accent))
    top_html = widgets.HTML(value=header + "".join(rows))

    if not breakdown:
        return widgets.VBox([top_html])

    breakdown_basis = max(
        (v for v in breakdown.values() if isinstance(v, (int, float))),
        default=0,
    )
    breakdown_rows = "".join(
        _stage_bar(name, val, breakdown_basis, accent="#26a69a")
        for name, val in breakdown.items()
    )
    breakdown_html = widgets.HTML(
        value=(
            "<div style='font-size:10.5px;color:var(--text-muted);"
            "padding:8px 0 4px;border-top:1px solid var(--border);margin-top:6px;'>"
            "Preprocess breakdown</div>"
            + breakdown_rows
        ),
        layout=widgets.Layout(display="none"),
    )

    toggle = widgets.Button(
        description="Show preprocess breakdown",
        icon="chevron-right",
        layout=widgets.Layout(width="220px", height="24px"),
    )

    state = {"expanded": False}

    def _on_toggle(_btn):
        state["expanded"] = not state["expanded"]
        breakdown_html.layout.display = "" if state["expanded"] else "none"
        toggle.icon = "chevron-down" if state["expanded"] else "chevron-right"
        toggle.description = (
            "Hide preprocess breakdown"
            if state["expanded"]
            else "Show preprocess breakdown"
        )

    toggle.on_click(_on_toggle)

    return widgets.VBox(
        [top_html, toggle, breakdown_html],
        layout=widgets.Layout(padding="4px 0 0 0"),
    )


def _render_unknown_schema(data, schema_version):
    raw = json.dumps(data, indent=2)
    warning = _diag_muted_card(
        f"Unknown <code>schema_version={schema_version}</code>; showing raw JSON."
    )
    pre = widgets.HTML(
        value=(
            f"<pre style='font-size:10.5px;background:var(--bg-input);"
            f"color:var(--text);padding:8px 10px;border-radius:4px;"
            f"border:1px solid var(--border);overflow:auto;max-height:240px;"
            f"white-space:pre-wrap;'>{raw}</pre>"
        )
    )
    return widgets.VBox([widgets.HTML(value=warning), pre])


def _render_diagnostics_file(path: Path):
    try:
        data = json.loads(path.read_text())
    except (OSError, ValueError) as exc:
        return widgets.HTML(value=_diag_error_card(f"Failed to read {path.name}: {exc}"))

    if not isinstance(data, dict):
        return widgets.HTML(
            value=_diag_error_card(f"{path.name}: expected JSON object, got {type(data).__name__}.")
        )

    schema_version = data.get("schema_version")
    if schema_version in _KNOWN_SCHEMA_VERSIONS:
        body = _render_known_schema(data)
    else:
        body = _render_unknown_schema(data, schema_version)

    title = widgets.HTML(
        value=(
            f"<div style='font-size:11.5px;font-weight:600;color:var(--text);"
            f"padding:6px 0 4px;'>{path.name}</div>"
        )
    )
    return widgets.VBox(
        [title, body],
        layout=widgets.Layout(
            padding="6px 10px",
            border="1px solid var(--border)",
            border_radius="4px",
            margin="0 0 6px 0",
        ),
    )


def build_diagnostics_panel(state):
    """Build the collapsible Pipeline-diagnostics panel.

    Returns a VBox that auto-hides when no sibling diagnostics JSON exists
    for the currently loaded series.
    """

    header_label = widgets.HTML(
        "<div class='nbpoc-section-label' style='padding-top:12px;'>"
        "PIPELINE DIAGNOSTICS</div>",
        layout=widgets.Layout(flex="1"),
    )
    toggle_btn = widgets.Button(
        icon="chevron-down",
        tooltip="Collapse / expand",
        layout=widgets.Layout(width="28px", height="22px"),
    )
    refresh_btn = widgets.Button(
        icon="refresh",
        tooltip="Rescan diagnostics files",
        layout=widgets.Layout(width="28px", height="22px"),
    )
    header = widgets.HBox(
        [header_label, refresh_btn, toggle_btn],
        layout=widgets.Layout(align_items="center", padding="0 0 6px"),
    )
    body_box = widgets.VBox([], layout=widgets.Layout(padding="4px 0"))

    container = widgets.VBox(
        [header, body_box],
        layout=widgets.Layout(
            display="none",
            padding="6px 0 0 0",
            margin="0",
        ),
    )

    state_flags = {"expanded": True}

    def _on_toggle(_btn):
        state_flags["expanded"] = not state_flags["expanded"]
        body_box.layout.display = "" if state_flags["expanded"] else "none"
        toggle_btn.icon = "chevron-down" if state_flags["expanded"] else "chevron-right"

    toggle_btn.on_click(_on_toggle)

    def _masks_dir_for_series():
        if not state.series_dir_path:
            return None
        return Path(state.series_dir_path).parent / _DIAG_MASKS_SUBDIR

    def _discover():
        masks_dir = _masks_dir_for_series()
        if masks_dir is None or not masks_dir.is_dir():
            container.layout.display = "none"
            body_box.children = []
            return

        files = sorted(
            p for p in masks_dir.glob(f"*{_DIAGNOSTICS_SUFFIX}") if p.is_file()
        )
        if not files:
            container.layout.display = "none"
            body_box.children = []
            return

        body_box.children = [_render_diagnostics_file(p) for p in files]
        container.layout.display = ""

    refresh_btn.on_click(lambda _b: _discover())
    state.observe(lambda _c: _discover(), names="series_dir_path")

    _discover()
    return container

# === File browser (utils/components/file_browser.py) ===================

"""File browser for DICOM images and series."""


from pathlib import Path

import ipywidgets as widgets


_NIFTI_MESSAGE_HTML = (
    "<div style='display:flex;align-items:center;justify-content:center;"
    "width:100%;min-height:350px;background:var(--bg-panel-alt);"
    "border-radius:8px;border:2px dashed var(--border-strong);"
    "color:var(--text-muted);font-size:13px;"
    "flex-direction:column;gap:8px;'>"
    "<span style='font-size:36px;opacity:0.5;'>&#129504;</span>"
    "<span>NIfTI files are not currently displayable.</span></div>"
)


def _fb_error_card(msg):
    return (
        f"<div style='color:var(--severity-severe-fg);"
        f"background:rgba(211,47,47,0.10);padding:8px 12px;"
        f"border-left:3px solid var(--severity-severe-fg);border-radius:4px;"
        f"font-size:11.5px;margin-top:4px;'>{msg}</div>"
    )


def _breadcrumb_html(directory) -> str:
    """Render the current directory as a breadcrumb trail.

    Takes the trailing path segments and renders them separated by ' / ' for
    visual parity with the clinical-workstation reference. The final segment
    is the 'current' crumb.
    """
    from pathlib import Path as _Path
    parts = list(_Path(str(directory)).parts)
    # Trim to last ~4 segments so the breadcrumb fits in the sidebar width
    trimmed = parts[-4:] if len(parts) > 4 else parts
    leading = "&hellip; / " if len(parts) > 4 else ""
    crumbs = []
    for i, part in enumerate(trimmed):
        cls = "crumb current" if i == len(trimmed) - 1 else "crumb"
        crumbs.append(f"<span class='{cls}'>{part}</span>")
    return (
        f"<div class='nbpoc-breadcrumb'>{leading}"
        + "<span class='sep'>/</span>".join(crumbs)
        + "</div>"
    )


def _build_browser(title, default_path, file_filter, file_icon, on_file_click,
                   on_clear=None, rows=15, extra_controls=None,
                   on_dir_change=None):
    """Build a generic file browser panel.

    Args:
        title: Section heading (may contain HTML entities).
        default_path: Initial directory to browse.
        file_filter: callable(Path) -> bool, which non-directory files to show.
        file_icon: Emoji string used for matching files.
        on_file_click: callable(Path) -> str|None.  Called when a matching
            file is selected.  Return an error message string to display,
            or *None* on success.
        on_clear: optional callable() invoked when the user clicks "Clear".
        rows: Number of visible rows in the Select widget.
        extra_controls: optional list of widgets appended to the controls HBox.
        on_dir_change: optional callback(Path) fired when the browsed directory
            changes.

    Returns:
        VBox widget for the browser panel.
    """

    _current_dir = [None]
    _refreshing = [False]

    root_text = widgets.Text(
        value=default_path,
        placeholder="/path/to/data",
        layout=widgets.Layout(width="100%"),
    )
    go_btn = widgets.Button(
        description="Go", icon="folder-open", button_style="info",
        layout=widgets.Layout(width="55px", height="30px"),
    )
    up_btn = widgets.Button(
        description="", icon="arrow-up",
        layout=widgets.Layout(width="34px", height="30px"),
    )
    clear_btn = widgets.Button(
        description="", icon="times",
        tooltip="Clear selection",
        layout=widgets.Layout(width="34px", height="30px"),
    )
    breadcrumb = widgets.HTML(
        value=(
            "<div class='nbpoc-breadcrumb'>"
            "<span class='crumb'>No directory selected</span></div>"
        ),
    )
    file_list = widgets.Select(
        options=[], rows=rows,
        layout=widgets.Layout(width="100%"),
    )
    status = widgets.HTML(value="")

    # ---- internal helpers ------------------------------------------------

    def _list_directory(directory):
        _refreshing[0] = True
        try:
            entries = sorted(
                directory.iterdir(),
                key=lambda p: (not p.is_dir(), p.name.lower()),
            )
        except PermissionError:
            status.value = _fb_error_card("Permission denied.")
            _refreshing[0] = False
            return

        options = []
        if directory.parent != directory:
            options.append("\u2B06 ..")
        for entry in entries:
            if entry.name.startswith("."):
                continue
            if entry.is_dir():
                options.append(f"\U0001F4C1 {entry.name}")
            elif file_filter(entry):
                options.append(f"{file_icon} {entry.name}")

        file_list.options = options if options else ["(empty)"]
        file_list.value = None
        _refreshing[0] = False

        breadcrumb.value = _breadcrumb_html(directory)

    def _on_go(_btn=None):
        p = Path(root_text.value.strip())
        if not p.exists() or not p.is_dir():
            status.value = _fb_error_card(f"Not found: {p}")
            return
        status.value = ""
        _current_dir[0] = p
        _list_directory(p)
        if on_dir_change:
            on_dir_change(p)

    def _on_up(_btn=None):
        if _current_dir[0] is None:
            return
        parent = _current_dir[0].parent
        if parent != _current_dir[0]:
            _current_dir[0] = parent
            root_text.value = str(parent)
            _list_directory(parent)
            if on_dir_change:
                on_dir_change(parent)

    def _on_select(change):
        if _refreshing[0]:
            return
        val = change["new"]
        if not val or val == "(empty)":
            return

        name = val.split(" ", 1)[1] if " " in val else val

        if name == "..":
            _on_up()
            return

        if _current_dir[0] is None:
            return
        target = _current_dir[0] / name

        if target.is_dir():
            _current_dir[0] = target
            root_text.value = str(target)
            _list_directory(target)
            if on_dir_change:
                on_dir_change(target)
        elif file_filter(target):
            error = on_file_click(target)
            if error:
                status.value = _fb_error_card(error)
            else:
                status.value = ""

    def _on_clear_click(_btn):
        if on_clear:
            on_clear()
        _refreshing[0] = True
        file_list.value = None
        _refreshing[0] = False
        status.value = ""

    # ---- wire events -----------------------------------------------------

    go_btn.on_click(_on_go)
    up_btn.on_click(_on_up)
    clear_btn.on_click(_on_clear_click)
    file_list.observe(_on_select, names="value")

    # Auto-browse on startup
    _on_go()

    bottom_controls = [go_btn, clear_btn]
    if extra_controls:
        bottom_controls.extend(extra_controls)

    title_html = widgets.HTML(
        f"<div class='nbpoc-section-label' style='padding:12px 12px 4px;'>"
        f"{title}</div>"
    )
    filter_row = widgets.VBox(
        [
            widgets.HBox(
                [root_text, up_btn],
                layout=widgets.Layout(width="100%", gap="4px"),
            ),
            widgets.HBox(
                bottom_controls,
                layout=widgets.Layout(width="100%", padding="6px 0 0 0", gap="4px"),
            ),
        ],
        layout=widgets.Layout(padding="0 12px 8px"),
    )

    list_box = widgets.VBox(
        [file_list],
        layout=widgets.Layout(padding="0 8px"),
    )

    panel = widgets.VBox(
        [title_html, breadcrumb, filter_row, list_box, status],
        layout=widgets.Layout(width="100%", padding="0"),
    )
    panel.add_class("nbpoc-sidebar")

    return panel


# =========================================================================
# Public builders
# =========================================================================

def build_image_browser(state, viewer):
    """Build DICOM image file browser (left sidebar)."""

    image_widget = viewer["image_widget"]
    image_placeholder = viewer["image_placeholder"]
    image_label = viewer["image_label"]
    metadata_html = viewer["metadata_html"]
    metadata_table = viewer["metadata_table"]
    series_nav = viewer["series_nav"]
    series_info_label = viewer["series_info_label"]

    _default_placeholder_html = image_placeholder.value

    def _clear_series_state():
        """Reset all series-related state and hide series UI."""
        state.series_datasets = []
        state.series_png_cache = []
        state.series_index = 0
        state.series_dir_name = ""
        state.series_dir_path = ""
        series_nav.layout.display = "none"

    def _on_image_selected(file_path):
        if is_nifti_file(file_path):
            _clear_series_state()
            state.current_ds = None
            state.current_png_bytes = None
            state.current_file_name = file_path.name
            state.current_file_path = ""
            image_widget.layout.display = "none"
            image_placeholder.value = _NIFTI_MESSAGE_HTML
            image_placeholder.layout.display = ""
            image_label.value = (
                f"<div style='font-size:12px;color:var(--text);padding:0 0 8px;'>"
                f"&#129504; <b>{file_path.name}</b></div>"
            )
            metadata_html.value = ""
            return None

        _clear_series_state()

        ds = read_dicom(file_path)
        if ds is None:
            return "Could not read DICOM file."

        try:
            _ = ds.pixel_array
        except Exception:
            return "No pixel data or missing transfer syntax handler."

        try:
            dicom_to_pil(ds)
        except Exception as exc:
            return f"Render error: {exc}"

        state.current_ds = ds
        state.current_png_bytes = dicom_to_png_bytes(ds)
        state.current_file_name = file_path.name
        state.current_file_path = str(file_path.resolve())

        # Show image, hide placeholder
        image_widget.value = state.current_png_bytes
        image_widget.layout.display = ""
        image_placeholder.layout.display = "none"
        image_label.value = (
            f"<div style='font-size:12px;color:var(--text);padding:0 0 8px;'>"
            f"&#x1F52C; <b>{file_path.name}</b></div>"
        )

        meta_rows = extract_metadata(ds)
        metadata_html.value = metadata_table(meta_rows) if meta_rows else ""

        return None

    def _on_image_clear():
        _clear_series_state()
        state.current_ds = None
        state.current_png_bytes = None
        state.current_file_name = ""
        state.current_file_path = ""
        image_widget.layout.display = "none"
        image_placeholder.value = _default_placeholder_html
        image_placeholder.layout.display = ""
        image_label.value = ""
        metadata_html.value = ""

    # -- Open Series button and wiring --

    open_series_btn = widgets.Button(
        description="Load Entire Series",
        icon="layer-group",
        button_style="warning",
        disabled=True,
        tooltip="Load all DICOMs in this directory as a series",
        layout=widgets.Layout(width="auto", min_width="150px", height="30px"),
    )

    _series_dir = [None]

    def _on_dir_change(directory):
        _series_dir[0] = directory
        try:
            count = sum(1 for p in directory.iterdir()
                        if is_dicom_candidate(p))
        except Exception:
            count = 0
        open_series_btn.disabled = count < 2

    def _on_open_series(_btn):
        if _series_dir[0] is None:
            return

        # Loading state
        open_series_btn.description = "Loading…"
        open_series_btn.disabled = True

        pairs = load_series(_series_dir[0])

        if not pairs:
            open_series_btn.description = "Series"
            open_series_btn.disabled = False
            return

        datasets = [ds for ds, _png in pairs]
        png_cache = [png for _ds, png in pairs]

        state.series_datasets = datasets
        state.series_png_cache = png_cache
        state.series_index = 0
        state.series_dir_name = _series_dir[0].name
        state.series_dir_path = str(_series_dir[0].resolve())

        # Set initial current slice
        state.current_ds = datasets[0]
        state.current_png_bytes = png_cache[0]
        state.current_file_name = f"{_series_dir[0].name} [1/{len(datasets)}]"
        state.current_file_path = ""

        # Show image, hide placeholder
        image_widget.value = png_cache[0]
        image_widget.layout.display = ""
        image_placeholder.layout.display = "none"
        image_label.value = (
            f"<div style='font-size:12px;color:var(--text);padding:0 0 8px;'>"
            f"&#x1F52C; <b>{_series_dir[0].name}</b>"
            f" &mdash; {len(datasets)} slices</div>"
        )

        # Show series navigation
        series_nav.layout.display = ""
        series_info_label.layout.display = ""
        series_info_label.value = (
            f"<div style='font-size:11.5px;color:var(--text-muted);padding:4px 0;"
            f"text-align:center;min-width:100px;'>"
            f"Slice 1 / {len(datasets)}</div>"
        )

        meta_rows = extract_metadata(datasets[0])
        metadata_html.value = metadata_table(meta_rows) if meta_rows else ""

        # Restore button
        open_series_btn.description = "Series"
        open_series_btn.disabled = False

    open_series_btn.on_click(_on_open_series)

    return _build_browser(
        title="&#x1F52C; Image Files",
        default_path=LOCAL_DATA_ROOT,
        file_filter=lambda p: is_dicom_candidate(p) or is_nifti_file(p),
        file_icon="\U0001F52C",
        on_file_click=_on_image_selected,
        on_clear=_on_image_clear,
        extra_controls=[open_series_btn],
        on_dir_change=_on_dir_change,
    )

# === Viewer tab (utils/components/viewer_tab.py) =======================

"""Image display, report display, and metadata panel."""


import ipywidgets as widgets
from ipyevents import Event


_VIEWER_PLACEHOLDER_HTML = (
    "<div style='display:flex;align-items:center;justify-content:center;"
    "width:100%;min-height:350px;background:transparent;"
    "color:var(--text-muted);font-size:13px;"
    "flex-direction:column;gap:8px;'>"
    "<span style='font-size:36px;opacity:0.4;'>&#128203;</span>"
    "<span>Select a series from the left sidebar</span></div>"
)


def _viewer_metadata_table(rows):
    trs = "".join(
        f"<tr><td style='padding:4px 12px;font-weight:600;white-space:nowrap;"
        f"color:var(--text);font-size:11.5px;"
        f"border-bottom:1px solid var(--border);'>{k}</td>"
        f"<td style='padding:4px 12px;font-size:11.5px;color:var(--text-muted);"
        f"border-bottom:1px solid var(--border);'>{v}</td></tr>"
        for k, v in rows
    )
    return (
        f"<table style='font-size:11.5px;border-collapse:collapse;width:100%;'>"
        f"{trs}</table>"
    )


def _viewer_fmt(value, suffix: str = "") -> str:
    """Format a DICOM scalar/sequence for an overlay (best-effort)."""
    if value is None or value == "":
        return "&mdash;"
    # DICOM multi-value (DSfloat, IS, etc.) is iterable; show first element.
    if hasattr(value, "__iter__") and not isinstance(value, (str, bytes)):
        try:
            value = next(iter(value))
        except StopIteration:
            return "&mdash;"
    try:
        if isinstance(value, float) or (
            isinstance(value, str) and "." in value
        ):
            return f"{float(value):.2f}{suffix}"
    except (TypeError, ValueError):
        pass
    return f"{value}{suffix}"


def build_viewer(state):
    """Build image display and metadata widgets.

    Returns a dict with keys:
        viewer_panel      - VBox for the main viewer area
        info_panel        - VBox for DICOM metadata (hidden until image loaded)
        image_widget      - Image widget (for cross-component updates)
        image_placeholder - HTML placeholder (hidden when content loaded)
        image_label       - HTML label showing filename
        metadata_html     - HTML widget for metadata table
        metadata_table    - formatter function for metadata rows
    """

    image_placeholder = widgets.HTML(value=_VIEWER_PLACEHOLDER_HTML)
    image_widget = widgets.Image(
        format="png",
        layout=widgets.Layout(
            max_width="100%", max_height="80vh", display="none",
            object_fit="contain",
        ),
    )
    # Wrapping the Image in a Box gives wheel/key events a reliable DOM target.
    # Events attached to <img> are inconsistent across JupyterLab versions.
    image_container = widgets.Box(
        [image_widget],
        layout=widgets.Layout(
            display="flex",
            justify_content="center",
            align_items="center",
            width="100%",
            min_height="500px",
            max_height="80vh",
            overflow="hidden",
        ),
    )
    # image_label and series_info_label are kept as no-op sinks for the
    # file_browser writes that still call into them. The corner overlays are
    # the user-visible labels now.
    image_label = widgets.HTML(value="", layout=widgets.Layout(display="none"))
    metadata_html = widgets.HTML(value="")

    series_info_label = widgets.HTML(
        value="",
        layout=widgets.Layout(display="none"),
    )

    # --- Corner overlays (top-left subject, top-right series, bottom-left
    # slice + W/L, bottom-right zoom + matrix). Absolutely positioned over the
    # image via the .nbpoc-viewer-canvas wrapper.
    overlay_tl = widgets.HTML(value="")
    overlay_tr = widgets.HTML(value="")
    overlay_bl = widgets.HTML(value="")
    overlay_br = widgets.HTML(value="")
    for ov, cls in (
        (overlay_tl, "tl"), (overlay_tr, "tr"),
        (overlay_bl, "bl"), (overlay_br, "br"),
    ):
        ov.add_class("nbpoc-viewer-overlay")
        ov.add_class(cls)

    def _update_overlays():
        ds = state.current_ds
        if ds is None:
            overlay_tl.value = overlay_tr.value = ""
            overlay_bl.value = overlay_br.value = ""
            return
        # Top-left: subject + study
        patient = str(
            getattr(ds, "PatientID", "") or getattr(ds, "PatientName", "") or ""
        )
        study_date = str(getattr(ds, "StudyDate", "") or "")
        study_desc = str(getattr(ds, "StudyDescription", "") or "")
        study_line = " ".join(s for s in (study_date, study_desc) if s)
        overlay_tl.value = (
            f"<div class='title'>{patient or '&mdash;'}</div>"
            f"<div class='muted'>{study_line or ''}</div>"
        )
        # Top-right: series description + number
        desc = str(getattr(ds, "SeriesDescription", "") or "")
        modality = str(getattr(ds, "Modality", "") or "")
        series_num = getattr(ds, "SeriesNumber", "")
        head_line = " ".join(s for s in (desc, modality) if s) or "&mdash;"
        overlay_tr.value = (
            f"<div class='title'>{head_line}</div>"
            f"<div class='muted'>Series: {_viewer_fmt(series_num)}</div>"
        )
        # Bottom-left: image i/n, location, thickness, W/L
        total = len(state.series_datasets)
        idx = state.series_index + 1 if total else 1
        loc = getattr(ds, "SliceLocation", None)
        thick = getattr(ds, "SliceThickness", None)
        ww = getattr(ds, "WindowWidth", None)
        wc = getattr(ds, "WindowCenter", None)
        overlay_bl.value = (
            f"<div>Image: {idx}{f' / {total}' if total else ''}</div>"
            f"<div>Loc: {_viewer_fmt(loc, ' mm')}</div>"
            f"<div>Thick: {_viewer_fmt(thick, ' mm')}</div>"
            f"<div>W: {_viewer_fmt(ww)} &nbsp; L: {_viewer_fmt(wc)}</div>"
        )
        # Bottom-right: zoom (stub) + matrix size
        rows = getattr(ds, "Rows", None)
        cols = getattr(ds, "Columns", None)
        overlay_br.value = (
            f"<div>Zoom: 100%</div>"
            f"<div>{_viewer_fmt(rows)} &times; {_viewer_fmt(cols)}</div>"
        )

    prev_btn = widgets.Button(
        description="", icon="arrow-up",
        tooltip="Previous slice",
        layout=widgets.Layout(width="32px", height="28px"),
    )
    next_btn = widgets.Button(
        description="", icon="arrow-down",
        tooltip="Next slice",
        layout=widgets.Layout(width="32px", height="28px"),
    )
    slice_slider = widgets.IntSlider(
        value=0, min=0, max=0, step=1,
        description="", readout=False, continuous_update=True,
        orientation="vertical",
        layout=widgets.Layout(height="240px", margin="0"),
    )
    _syncing_slider = [False]
    # Vertical scrub stack — prev / slider / next, absolutely positioned on
    # the right edge of the canvas via the .nbpoc-viewer-slider CSS class.
    series_nav = widgets.VBox(
        [prev_btn, slice_slider, next_btn],
        layout=widgets.Layout(display="none", align_items="center"),
    )
    series_nav.add_class("nbpoc-viewer-slider")

    def _go_to_slice(idx):
        """Navigate to a specific slice index, updating all state and UI."""
        total = len(state.series_datasets)
        if total == 0 or idx < 0 or idx >= total:
            return
        state.series_index = idx
        ds, png = state.series_datasets[idx], state.series_png_cache[idx]
        state.current_ds = ds
        state.current_png_bytes = png
        state.current_file_name = f"{state.series_dir_name} [{idx + 1}/{total}]"

        image_widget.value = png
        series_info_label.value = (
            f"<div style='font-size:12px;color:#6c757d;padding:4px 0;"
            f"text-align:center;min-width:100px;'>"
            f"Slice {idx + 1} / {total}</div>"
        )

        if slice_slider.value != idx:
            _syncing_slider[0] = True
            try:
                slice_slider.value = idx
            finally:
                _syncing_slider[0] = False

        meta_rows = extract_metadata(ds)
        if meta_rows:
            metadata_html.value = _viewer_metadata_table(meta_rows)
        _update_overlays()

    def _on_prev(_btn):
        _go_to_slice(state.series_index - 1)

    def _on_next(_btn):
        _go_to_slice(state.series_index + 1)

    def _on_slider(change):
        if _syncing_slider[0]:
            return
        _go_to_slice(int(change["new"]))

    def _on_series_datasets_change(_change):
        total = len(state.series_datasets)
        _syncing_slider[0] = True
        try:
            slice_slider.max = max(0, total - 1)
            slice_slider.value = min(state.series_index, max(0, total - 1))
        finally:
            _syncing_slider[0] = False

    prev_btn.on_click(_on_prev)
    next_btn.on_click(_on_next)
    slice_slider.observe(_on_slider, names="value")
    state.observe(_on_series_datasets_change, names="series_datasets")
    state.observe(lambda _c: _update_overlays(), names="current_ds")

    # Scroll-wheel + arrow-key nav via ipyevents. Attach to BOTH the Box and
    # the Image — the Box catches bubbled events in JupyterLab, but Voila and
    # some browsers route the wheel directly to the <img> and the Box never
    # sees it. Two sources, same handler.
    _wheel_box = Event(
        source=image_container,
        watched_events=["wheel"],
        prevent_default_action=True,
    )
    _wheel_img = Event(
        source=image_widget,
        watched_events=["wheel"],
        prevent_default_action=True,
    )
    _key_event = Event(
        source=image_container,
        watched_events=["keydown"],
    )

    def _on_wheel(event):
        dy = event.get("deltaY", 0)
        if dy > 0:
            _go_to_slice(state.series_index + 1)
        elif dy < 0:
            _go_to_slice(state.series_index - 1)

    def _on_key(event):
        key = event.get("key", "")
        if key in ("ArrowDown", "ArrowRight", "PageDown", "j"):
            _go_to_slice(state.series_index + 1)
        elif key in ("ArrowUp", "ArrowLeft", "PageUp", "k"):
            _go_to_slice(state.series_index - 1)

    _wheel_box.on_dom_event(_on_wheel)
    _wheel_img.on_dom_event(_on_wheel)
    _key_event.on_dom_event(_on_key)

    canvas = widgets.Box(
        [
            image_placeholder,
            image_container,
            overlay_tl,
            overlay_tr,
            overlay_bl,
            overlay_br,
            series_nav,
        ],
        layout=widgets.Layout(width="100%", flex="1"),
    )
    canvas.add_class("nbpoc-viewer-canvas")

    viewer_panel = widgets.VBox(
        [image_label, canvas],
        layout=widgets.Layout(width="100%", height="100%"),
    )
    viewer_panel.add_class("nbpoc-viewer")

    info_panel = widgets.VBox(
        [
            widgets.HTML(
                "<div class='nbpoc-section-label' "
                "style='padding:8px 0 4px;border-top:1px solid var(--border);"
                "margin-top:12px;'>DICOM METADATA</div>"
            ),
            metadata_html,
        ],
        layout=widgets.Layout(padding="0", width="100%", display="none"),
    )

    return {
        "viewer_panel": viewer_panel,
        "info_panel": info_panel,
        "image_widget": image_widget,
        "image_container": image_container,
        "image_placeholder": image_placeholder,
        "image_label": image_label,
        "metadata_html": metadata_html,
        "metadata_table": _viewer_metadata_table,
        "series_nav": series_nav,
        "series_info_label": series_info_label,
        "go_to_slice": _go_to_slice,
        "_wheel_box": _wheel_box,
        "_wheel_img": _wheel_img,
        "_key_event": _key_event,
    }

# === Segmentation tab (utils/components/segmentation_tab.py) ===========

"""Segmentation panel for the KServe TotalSegmentator endpoint."""


import time
from pathlib import Path

import ipywidgets as widgets
import pydicom
import requests


# InferenceService names. Payload shape varies: TotalSegmentator uses
# fast/roi_subset, NSCLC uses output_format/threshold/mask_tag, BrainSeg
# uses four modality dirs (t1/t2/t1ce/flair).
NSCLC_MODEL = "duneai-nsclc"
BRAINSEG_MODEL = "brainseg"

# Dropdown label -> InferenceService name. The name is substituted into
# INFERENCE_URL_TEMPLATE by http_client.predict().
TASKS: dict[str, str] = {
    "nsclc_segmentation": NSCLC_MODEL,
    "brainseg_segmentation": BRAINSEG_MODEL,
    "TotalSegmentator": "totalseg",
}

_BRAINSEG_MODALITIES = ("t1", "t1ce", "t2", "flair")
_BRAINSEG_MODALITY_LABELS = {
    "t1": "T1",
    "t1ce": "T1ce (post)",
    "t2": "T2",
    "flair": "FLAIR",
}

_SEGTAB_SPINNER_HTML = (
    '<div style="display:flex;align-items:center;gap:8px;padding:8px 0;">'
    '<span class="nbpoc-spinner"></span>'
    '<span style="font-size:12px;color:var(--text-muted);">'
    'Running inference&hellip;</span></div>'
)

_SEGTAB_PLACEHOLDER = (
    "<div style='color:var(--text-muted);padding:16px 4px;text-align:center;"
    "font-size:12px;'>Select a DICOM series and click "
    "<b>Analyze Current Slice</b> to generate a result.</div>"
)


def _segtab_error_card(msg: str) -> str:
    return (
        f"<div class='nbpoc-card severe' style='font-size:12px;color:var(--severity-severe-fg);'>"
        f"{msg}</div>"
    )


def _segtab_response_card(result: dict, local_seg_path: str | None) -> str:
    # Show whichever fields the predictor echoed back; TotalSegmentator and
    # NSCLC return different keys.
    rows = []
    if local_seg_path:
        rows.append(
            f"<div style='word-break:break-all;'><b>seg_path</b> &nbsp; "
            f"{local_seg_path}</div>"
        )
    for key in ("task", "fast", "output_format", "threshold"):
        val = result.get(key)
        if val is not None:
            rows.append(f"<div><b>{key}</b> &nbsp; {val}</div>")
    roi = result.get("roi_subset")
    if roi:
        rows.append(f"<div><b>roi_subset</b> &nbsp; {', '.join(roi)}</div>")
    return (
        "<div class='nbpoc-card normal' "
        "style='font-size:11px;font-family:monospace;line-height:1.55;'>"
        "<div style='font-weight:700;color:var(--text);margin-bottom:6px;"
        "font-family:inherit;'>"
        "Inference Result</div>"
        + "".join(rows) +
        "</div>"
    )


def _guess_brainseg_modality(series_desc: str) -> str | None:
    """Best-effort SeriesDescription -> {t1, t1ce, t2, flair}. None if unsure."""
    s = (series_desc or "").lower()
    if "flair" in s:
        return "flair"
    if "t2" in s:
        return "t2"
    if "t1" in s:
        post_cues = ("post", "+c", "stealth-post", "contrast", " ce", "_ce")
        gd_cue = "gd"  # word-bounded check below to avoid matching e.g. "edge"
        if any(c in s for c in post_cues) or any(
            tok == gd_cue for tok in s.replace("+", " ").replace("_", " ").split()
        ):
            return "t1ce"
        return "t1"
    return None


def _discover_brainseg_scans(series_dir_path: str) -> dict[str, dict]:
    """Find sibling scans under the XNAT SCANS/ root of the selected series.

    Returns a dict keyed by scan ID (the dir name under SCANS/), each value
    {path, series_desc, guess}. ``path`` is the DICOM dir (prefers SCANS/<id>/
    DICOM/ if present, else SCANS/<id>/) — i.e. what we'll send to the
    transformer after translate_path().
    """
    if not series_dir_path:
        return {}
    p = Path(series_dir_path)
    scans_root: Path | None = None
    for ancestor in [p, *p.parents]:
        if ancestor.name == "SCANS":
            scans_root = ancestor
            break
    if scans_root is None or not scans_root.is_dir():
        return {}

    result: dict[str, dict] = {}
    for child in sorted(scans_root.iterdir(), key=lambda c: c.name):
        if not child.is_dir():
            continue
        dicom_dir = child / "DICOM" if (child / "DICOM").is_dir() else child
        series_desc = ""
        try:
            for f in dicom_dir.iterdir():
                if f.is_file() and not f.name.startswith("."):
                    try:
                        ds = pydicom.dcmread(
                            str(f), stop_before_pixels=True, force=True
                        )
                        series_desc = str(getattr(ds, "SeriesDescription", "") or "")
                    except Exception:
                        series_desc = ""
                    break
        except OSError:
            continue
        result[child.name] = {
            "path": str(dicom_dir),
            "series_desc": series_desc,
            "guess": _guess_brainseg_modality(series_desc),
        }
    return result


def build_segmentation(state):
    """Build the segmentation request panel (TotalSegmentator)."""

    task_labels = list(TASKS.keys())
    task_dropdown = widgets.Dropdown(
        options=task_labels,
        value=task_labels[0],
        description="Task:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "50px"},
    )

    series_label = widgets.HTML(value="")

    fast_checkbox = widgets.Checkbox(
        value=True, description="Fast (3 mm, ~20–30 s)", indent=False,
    )
    fast_checkbox_bar = widgets.HBox([fast_checkbox])
    fast_checkbox_bar.add_class("medgemma-switch")

    roi_input = widgets.Text(
        value="",
        placeholder="Optional: liver, heart, aorta",
        description="ROI subset:",
        style={"description_width": "90px"},
        layout=widgets.Layout(width="100%"),
    )

    # NSCLC-only control: tumor probability cutoff sent as `threshold`.
    threshold_input = widgets.BoundedFloatText(
        value=0.99,
        min=0.0,
        max=1.0,
        step=0.01,
        description="Threshold:",
        style={"description_width": "90px"},
        layout=widgets.Layout(width="100%"),
    )

    mask_tag_input = widgets.Text(
        value="",
        placeholder="Optional: high_thresh_v1",
        description="Mask tag:",
        style={"description_width": "90px"},
        layout=widgets.Layout(width="100%"),
    )

    # BrainSeg requires four co-registered modalities — one dropdown per
    # modality, populated from sibling scan dirs under .../SCANS/ of whatever
    # series is currently loaded in the file browser.
    brainseg_dropdowns: dict[str, widgets.Dropdown] = {}
    for key in _BRAINSEG_MODALITIES:
        brainseg_dropdowns[key] = widgets.Dropdown(
            options=[("(load a series first)", None)],
            value=None,
            description=f"{_BRAINSEG_MODALITY_LABELS[key]}:",
            layout=widgets.Layout(width="100%"),
            style={"description_width": "90px"},
        )
    brainseg_status = widgets.HTML(value="")
    brainseg_block = widgets.VBox(
        [brainseg_status, *(brainseg_dropdowns[k] for k in _BRAINSEG_MODALITIES)]
    )
    _brainseg_scans_cache: dict[str, dict] = {}

    run_button = widgets.Button(
        description="Analyze Current Slice",
        icon="bolt",
        button_style="primary",
        layout=widgets.Layout(width="100%", height="38px"),
    )
    run_button.add_class("nbpoc-analyze-wrap")

    spinner = widgets.HTML(value="")
    response_area = widgets.HTML(value=_SEGTAB_PLACEHOLDER)

    def _refresh_series_label(*_):
        n = len(state.series_datasets)
        if state.series_dir_name and n > 0:
            series_label.value = (
                f"<div style='font-size:11.5px;color:var(--text-muted);padding:4px 0;'>"
                f"<b style='color:var(--text);'>Series:</b> {state.series_dir_name} "
                f"({n} slice{'s' if n != 1 else ''})</div>"
            )
            run_button.disabled = False
        else:
            series_label.value = (
                "<div style='font-size:11.5px;color:var(--warn-fg);padding:4px 0;'>"
                "Load a DICOM series to enable inference."
                "</div>"
            )
            run_button.disabled = True

    state.observe(
        _refresh_series_label, names=["series_dir_name", "series_datasets"]
    )
    _refresh_series_label()

    def _apply_task_visibility(*_):
        """Show only the controls relevant to the selected task."""
        model = TASKS[task_dropdown.value]
        is_nsclc = model == NSCLC_MODEL
        is_brainseg = model == BRAINSEG_MODEL
        is_totalseg = not (is_nsclc or is_brainseg)
        fast_checkbox_bar.layout.display = "" if is_totalseg else "none"
        roi_input.layout.display = "" if is_totalseg else "none"
        threshold_input.layout.display = "" if is_nsclc else "none"
        mask_tag_input.layout.display = "" if is_nsclc else "none"
        brainseg_block.layout.display = "" if is_brainseg else "none"

    task_dropdown.observe(_apply_task_visibility, names="value")
    _apply_task_visibility()

    def _refresh_brainseg_scans(*_):
        scans = _discover_brainseg_scans(state.series_dir_path)
        _brainseg_scans_cache.clear()
        _brainseg_scans_cache.update(scans)

        if scans:
            opts = [("(choose scan)", None)] + [
                (
                    f"{sid} — {info['series_desc'] or '(no description)'}",
                    sid,
                )
                for sid, info in scans.items()
            ]
        else:
            opts = [("(no SCANS/ sibling dir found)", None)]

        for key, dd in brainseg_dropdowns.items():
            dd.options = opts
            default = next(
                (sid for sid, info in scans.items() if info["guess"] == key), None
            )
            dd.value = default

        if not scans:
            brainseg_status.value = (
                "<div style='font-size:11.5px;color:var(--warn-fg);padding:4px 0;'>"
                "Load a series under <code>.../SCANS/&lt;id&gt;/DICOM/</code> "
                "to populate modality choices.</div>"
            )
        else:
            unmatched = [
                _BRAINSEG_MODALITY_LABELS[k]
                for k, dd in brainseg_dropdowns.items()
                if dd.value is None
            ]
            tail = (
                f" Could not auto-detect: {', '.join(unmatched)}."
                if unmatched
                else " Auto-detected — verify before running."
            )
            brainseg_status.value = (
                f"<div style='font-size:11.5px;color:var(--text-muted);padding:4px 0;'>"
                f"Found {len(scans)} sibling scan{'s' if len(scans) != 1 else ''}.{tail}"
                "</div>"
            )

    state.observe(_refresh_brainseg_scans, names="series_dir_path")
    _refresh_brainseg_scans()

    def _parse_roi(value: str) -> list[str] | None:
        items = [s.strip() for s in value.split(",") if s.strip()]
        return items or None

    def _build_payload() -> tuple[dict | None, str | None]:
        if not state.series_dir_path:
            return None, "No series loaded — pick one from the file browser first."
        try:
            dicom_dir = translate_path(state.series_dir_path)
        except ValueError as e:
            return None, str(e)

        if TASKS[task_dropdown.value] == NSCLC_MODEL:
            payload: dict = {
                "dicom_dir": dicom_dir,
                "output_format": "dicom",
                "threshold": float(threshold_input.value),
            }
            tag = mask_tag_input.value.strip()
            if tag:
                payload["mask_tag"] = tag
            return payload, None

        if TASKS[task_dropdown.value] == BRAINSEG_MODEL:
            if not _brainseg_scans_cache:
                return None, (
                    "No sibling scans found. Load a series under "
                    ".../SCANS/<id>/DICOM/ first."
                )
            selected = {k: brainseg_dropdowns[k].value for k in _BRAINSEG_MODALITIES}
            missing = [
                _BRAINSEG_MODALITY_LABELS[k] for k, v in selected.items() if not v
            ]
            if missing:
                return None, f"Pick a scan for: {', '.join(missing)}"
            if len(set(selected.values())) != len(selected):
                return None, "Each modality must map to a different scan."
            payload = {"output_format": "dicom"}
            for key, scan_id in selected.items():
                local_path = _brainseg_scans_cache[scan_id]["path"]
                try:
                    payload[f"{key}_dir"] = translate_path(local_path)
                except ValueError as e:
                    return None, str(e)
            return payload, None

        payload = {"dicom_dir": dicom_dir, "fast": fast_checkbox.value}
        roi = _parse_roi(roi_input.value)
        if roi:
            payload["roi_subset"] = roi
        return payload, None

    def _on_run(_btn):
        payload, err = _build_payload()
        if err:
            response_area.value = _segtab_error_card(err)
            return

        model_name = TASKS[task_dropdown.value]
        run_button.disabled = True
        spinner.value = _SEGTAB_SPINNER_HTML
        t0 = time.time()

        try:
            result = predict(payload, model_name)
            elapsed = time.time() - t0

            local_seg = None
            seg_path = result.get("seg_path")
            if seg_path:
                try:
                    local_seg = translate_pod_path_to_local(seg_path)
                except ValueError as e:
                    response_area.value = _segtab_error_card(
                        f"Segmentation succeeded but seg_path could not be "
                        f"translated: {e}"
                    )
                    return

            predictor_elapsed = float(result.get("elapsed_s", 0.0))
            footer = (
                f"<div style='font-size:10.5px;color:var(--text-dim);margin-top:8px;'>"
                f"Response: {elapsed:.1f}s &nbsp;&middot;&nbsp; "
                f"Predictor: {predictor_elapsed:.1f}s &nbsp;&middot;&nbsp; "
                f"Refresh the results section to load the new mask.</div>"
            )
            response_area.value = _segtab_response_card(result, local_seg) + footer

        except requests.Timeout:
            response_area.value = _segtab_error_card(
                "Request timed out. KServe may be cold-starting "
                "(can take several minutes on first request)."
            )
        except requests.HTTPError as e:
            body = ""
            try:
                body = e.response.text[:500]
            except Exception:
                pass
            response_area.value = _segtab_error_card(
                f"HTTP {e.response.status_code} from predictor: {body}"
            )
        except requests.RequestException as e:
            response_area.value = _segtab_error_card(f"Request failed: {e}")
        except Exception as e:
            response_area.value = _segtab_error_card(f"Unexpected error: {e}")
        finally:
            run_button.disabled = not state.series_dir_path
            spinner.value = ""

    run_button.on_click(_on_run)

    return widgets.VBox(
        [
            task_dropdown,
            series_label,
            fast_checkbox_bar,
            roi_input,
            threshold_input,
            mask_tag_input,
            brainseg_block,
            run_button,
            spinner,
            response_area,
        ],
        layout=widgets.Layout(width="100%", padding="0"),
    )

# === Seg viewer (utils/components/seg_viewer.py) =======================

"""Auto-discovering masks panel for the loaded DICOM series.

On series load, scans ``{state.series_dir_path}/../masks/*.dcm`` for DICOM SEG
files and exposes one checkbox per file. Multiple masks can be enabled at
once; they composite in checkbox order (later masks layer on top).

SEGs with more than one labeled segment expand to a per-segment checkbox list
so individual structures can be hidden. The opacity slider and orientation
buttons (rotate, flip H, flip V) apply to every active overlay.
"""


import io
from pathlib import Path

import ipywidgets as widgets
import numpy as np
from PIL import Image



_SV_MASKS_SUBDIR = "masks"


def _sv_pil_to_png_bytes(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG", compress_level=1)
    return buf.getvalue()


def _sv_error_card(msg):
    return (
        f"<div class='nbpoc-card severe' style='font-size:11.5px;"
        f"color:var(--severity-severe-fg);'>{msg}</div>"
    )


def _sv_muted_card(msg):
    return (
        f"<div style='color:var(--text-muted);background:var(--bg-panel-alt);"
        f"padding:10px 12px;border-left:3px solid var(--border-strong);"
        f"border-radius:4px;font-size:11.5px;line-height:1.4;'>{msg}</div>"
    )


def _sv_color_swatch(rgb):
    r, g, b = rgb
    return (
        f"<span style='display:inline-block;width:11px;height:11px;"
        f"background:rgb({r},{g},{b});border-radius:2px;"
        f"border:1px solid rgba(255,255,255,0.15);vertical-align:middle;'></span>"
    )


def _sv_severity_for_seg(num_segments: int) -> str:
    """Bucket: 0 → normal, 1-2 → moderate, 3+ → severe. Tweakable."""
    if num_segments <= 0:
        return "normal"
    if num_segments <= 2:
        return "moderate"
    return "severe"


def build_seg_viewer(state, viewer):
    """Build the auto-discovering masks panel.

    Rebuilds the mask list whenever ``state.series_dir_path`` changes. Each
    discovered ``.dcm`` becomes a checkbox; toggling drives a recomposite of
    the series PNG cache so the viewer reflects the active overlay set.
    """

    image_widget = viewer["image_widget"]

    # Pristine snapshots of the per-slice PNGs at series-load time. We never
    # mutate this; recomposites always start from base_arrays (decoded once)
    # and write into state.series_png_cache.
    _originals: list[bytes] | None = None
    _base_arrays: list[np.ndarray] | None = None

    # Per-mask state. List ordering is the composite z-order (top of list
    # paints first, later entries layer on top).
    # Entry shape:
    #   {
    #     "path": str,
    #     "name": str,
    #     "parsed": dict | None,        # cached load_dicom_seg result
    #     "parse_error": str | None,
    #     "row": HBox,                  # the checkbox/expand row
    #     "checkbox": Checkbox,         # master enable
    #     "expand_btn": Button,
    #     "segments_box": VBox,         # per-segment checkbox container
    #     "segment_checkboxes": {segment_number: Checkbox},
    #     "expanded": bool,
    #   }
    _masks: list[dict] = []

    # Orientation, applied to every mask before compositing so all overlays
    # share the same axis convention. Identical semantics to the prior
    # rotate/flip controls.
    _rotation = [0]  # number of 90° CCW turns
    _flip_h = [False]
    _flip_v = [False]

    # --- widgets ------------------------------------------------------------

    header_label = widgets.HTML(
        "<div class='nbpoc-section-label'>RESULTS (0)</div>",
        layout=widgets.Layout(flex="1"),
    )
    refresh_btn = widgets.Button(
        icon="refresh",
        tooltip="Rescan the masks folder",
        layout=widgets.Layout(width="28px", height="24px"),
    )
    header = widgets.HBox(
        [header_label, refresh_btn],
        layout=widgets.Layout(align_items="center", padding="12px 0 4px"),
    )
    status_html = widgets.HTML(value="")
    mask_list_box = widgets.VBox([], layout=widgets.Layout(padding="4px 0"))

    alpha_slider = widgets.FloatSlider(
        value=0.4, min=0.1, max=0.9, step=0.05,
        description="Opacity:", continuous_update=False,
        style={"description_width": "60px"},
        layout=widgets.Layout(width="100%"),
    )
    rotate_btn = widgets.Button(
        description="Rotate 90°", icon="redo",
        tooltip="Rotate overlays 90° CCW (cycles 0/90/180/270)",
        layout=widgets.Layout(width="100px", height="28px"),
    )
    flip_h_btn = widgets.Button(
        description="Flip H", icon="arrows-h",
        tooltip="Mirror overlays horizontally",
        layout=widgets.Layout(width="74px", height="28px"),
    )
    flip_v_btn = widgets.Button(
        description="Flip V", icon="arrows-v",
        tooltip="Mirror overlays vertically",
        layout=widgets.Layout(width="74px", height="28px"),
    )
    orient_row = widgets.HBox(
        [rotate_btn, flip_h_btn, flip_v_btn],
        layout=widgets.Layout(gap="6px", padding="4px 0"),
    )
    display_controls_label = widgets.HTML(
        "<div class='nbpoc-section-label' style='padding-top:12px;'>DISPLAY</div>"
    )

    # --- helpers ------------------------------------------------------------

    def _refresh_viewer_image():
        if state.series_datasets and state.series_png_cache:
            idx = state.series_index
            if 0 <= idx < len(state.series_png_cache):
                image_widget.value = state.series_png_cache[idx]

    def _ensure_base_arrays():
        nonlocal _originals, _base_arrays
        if _originals is None and state.series_png_cache:
            _originals = list(state.series_png_cache)
        if _originals is None:
            return False
        if _base_arrays is None or len(_base_arrays) != len(_originals):
            _base_arrays = [
                np.array(
                    Image.open(io.BytesIO(png)).convert("RGB"),
                    dtype=np.float32,
                )
                for png in _originals
            ]
        return True

    def _reset_caches():
        nonlocal _originals, _base_arrays
        _originals = None
        _base_arrays = None

    def _transform_mask_array(arr):
        if _flip_h[0]:
            arr = np.fliplr(arr)
        if _flip_v[0]:
            arr = np.flipud(arr)
        if _rotation[0]:
            arr = np.rot90(arr, _rotation[0])
        return arr

    def _active_layers():
        """Return list of (by_sop, segments, segments_enabled) for enabled masks."""
        layers = []
        for entry in _masks:
            if not entry["checkbox"].value:
                continue
            parsed = entry["parsed"]
            if parsed is None:
                continue
            enabled_segs = {
                num for num, cb in entry["segment_checkboxes"].items() if cb.value
            }
            if not enabled_segs:
                continue
            layers.append((parsed["by_source_sop"], parsed["segments"], enabled_segs))
        return layers

    def _recomposite():
        """Rebuild state.series_png_cache from base arrays + active layers."""
        if not state.series_datasets:
            return
        if not _ensure_base_arrays():
            return
        layers = _active_layers()
        alpha = float(alpha_slider.value)

        if not layers:
            state.series_png_cache = list(_originals)
            _refresh_viewer_image()
            return

        new_cache = list(_originals)
        for idx, ds in enumerate(state.series_datasets):
            sop = getattr(ds, "SOPInstanceUID", None)
            if not sop:
                continue
            sop_key = str(sop)

            base = _base_arrays[idx]
            composite = None
            for by_sop, segments, enabled_segs in layers:
                slice_masks = by_sop.get(sop_key)
                if not slice_masks:
                    continue
                filtered = {
                    n: slice_masks[n]
                    for n in enabled_segs
                    if n in slice_masks
                }
                if not filtered:
                    continue
                if composite is None:
                    composite = base.copy()
                h, w = composite.shape[:2]
                for seg_num, mask in filtered.items():
                    mask = _transform_mask_array(mask)
                    if mask.shape != (h, w):
                        mask_pil = Image.fromarray(
                            (mask.astype(np.uint8) * 255), mode="L"
                        ).resize((w, h), Image.NEAREST)
                        mask = np.array(mask_pil) > 127
                    color = np.array(segments[seg_num]["color"], dtype=np.float32)
                    composite[mask] = (
                        (1.0 - alpha) * composite[mask] + alpha * color
                    )
            if composite is not None:
                img = Image.fromarray(
                    np.clip(composite, 0, 255).astype(np.uint8), mode="RGB"
                )
                new_cache[idx] = _sv_pil_to_png_bytes(img)

        state.series_png_cache = new_cache
        _refresh_viewer_image()

    def _restore_originals():
        if _originals is None:
            return
        state.series_png_cache = list(_originals)
        _refresh_viewer_image()

    # --- mask discovery + UI build -----------------------------------------

    def _masks_dir_for_series():
        if not state.series_dir_path:
            return None
        series_dir = Path(state.series_dir_path)
        return series_dir.parent / _SV_MASKS_SUBDIR

    def _build_segment_row(entry, num, label, color, slice_count):
        cb = widgets.Checkbox(
            value=True,
            description="",
            indent=False,
            layout=widgets.Layout(width="22px", margin="0"),
        )
        cb.observe(lambda _c: _recomposite(), names="value")
        entry["segment_checkboxes"][num] = cb
        meta = widgets.HTML(
            f"<div style='font-size:12px;color:#495057;line-height:1.4;'>"
            f"{_sv_color_swatch(color)} &nbsp;{label}"
            f"<span style='color:#6c757d;font-size:11px;'> &nbsp;&middot;&nbsp; "
            f"{slice_count} slice{'s' if slice_count != 1 else ''}</span></div>"
        )
        return widgets.HBox(
            [cb, meta],
            layout=widgets.Layout(
                align_items="center",
                padding="2px 0 2px 18px",
            ),
        )

    def _populate_segments(entry):
        parsed = entry["parsed"]
        seg_rows = []
        segments = parsed["segments"]
        matched_counts: dict[int, int] = {}
        for sop_masks in parsed["by_source_sop"].values():
            for num in sop_masks:
                matched_counts[num] = matched_counts.get(num, 0) + 1
        for num in sorted(segments.keys()):
            info = segments[num]
            seg_rows.append(
                _build_segment_row(
                    entry,
                    num=num,
                    label=info["label"],
                    color=info["color"],
                    slice_count=matched_counts.get(num, 0),
                )
            )
        entry["segments_box"].children = seg_rows

    def _on_expand_clicked(entry):
        def _handler(_btn):
            entry["expanded"] = not entry["expanded"]
            if entry["expanded"]:
                if entry["parsed"] is None and not entry["parse_error"]:
                    _ensure_parsed(entry)
                if entry["parsed"] is not None and not entry["segment_checkboxes"]:
                    _populate_segments(entry)
                if entry["parsed"] is not None:
                    _refresh_card_severity(entry)
            target = entry.get("drawer") or entry["segments_box"]
            target.layout.display = "" if entry["expanded"] else "none"
        return _handler

    def _refresh_card_severity(entry):
        """Recompute card severity from parsed segment count + restyle."""
        parsed = entry.get("parsed")
        title_html = entry.get("title_html")
        card = entry.get("card")
        if parsed is None or title_html is None or card is None:
            return
        seg_count = len(parsed.get("segments") or {})
        severity = _sv_severity_for_seg(seg_count)
        for cls in ("normal", "moderate", "severe"):
            card.remove_class(cls)
        card.add_class(severity)
        title_html.value = (
            f"<div class='nbpoc-card-head'>"
            f"<div class='nbpoc-card-title'>{Path(entry['path']).stem}</div>"
            f"<span class='nbpoc-card-severity {severity}'>{severity}</span>"
            f"</div>"
            f"<div class='nbpoc-card-meta'>"
            f"<span>See report</span>"
            f"<span class='sep'>&middot;</span>"
            f"<span>{seg_count} segment{'s' if seg_count != 1 else ''}</span>"
            f"<span class='sep'>&middot;</span>"
            f"<span>50%</span>"
            f"</div>"
        )

    def _ensure_parsed(entry):
        if entry["parsed"] is not None or entry["parse_error"]:
            return
        try:
            entry["parsed"] = load_dicom_seg(Path(entry["path"]))
        except Exception as e:
            entry["parse_error"] = str(e)
            entry["checkbox"].disabled = True
            entry["checkbox"].description = f"{entry['name']}  (not a SEG: {e})"

    def _on_mask_toggle(entry):
        def _handler(change):
            if change["new"]:
                _ensure_parsed(entry)
                if entry["parse_error"]:
                    return
                if not entry["segment_checkboxes"]:
                    _populate_segments(entry)
            _recomposite()
        return _handler

    def _build_mask_entry(path: Path) -> dict:
        entry: dict = {
            "path": str(path),
            "name": path.name,
            "parsed": None,
            "parse_error": None,
            "segment_checkboxes": {},
            "expanded": False,
        }
        # Master enable for the overlay — lives alongside Accept/Edit/Reject
        # so the user can toggle visibility without opening the Edit drawer.
        checkbox = widgets.Checkbox(
            value=False,
            description="Show",
            indent=False,
            tooltip="Toggle this mask's overlay on the viewer",
            layout=widgets.Layout(width="auto"),
        )
        # Per-segment toggles, populated lazily once the SEG is parsed.
        segments_box = widgets.VBox([], layout=widgets.Layout(display="none"))

        # Action row: Accept / Edit / Reject. Accept is a no-op flag, Edit
        # toggles the Edit drawer (enable checkbox + segments box), Reject
        # hides the card and disables any active overlay for that mask.
        accept_btn = widgets.Button(
            description="Accept", icon="check",
            layout=widgets.Layout(width="auto", height="26px"),
        )
        edit_btn = widgets.Button(
            description="Edit", icon="pencil",
            layout=widgets.Layout(width="auto", height="26px"),
        )
        reject_btn = widgets.Button(
            description="Reject", icon="times",
            layout=widgets.Layout(width="auto", height="26px"),
        )
        for b in (accept_btn, edit_btn, reject_btn):
            b.add_class("nbpoc-card-action")
        accept_btn.add_class("accept")
        edit_btn.add_class("edit")
        reject_btn.add_class("reject")
        action_row = widgets.HBox(
            [checkbox, accept_btn, edit_btn, reject_btn],
            layout=widgets.Layout(gap="6px", padding="0"),
        )
        action_row.add_class("nbpoc-card-actions")

        # The Edit drawer holds per-segment toggles; hidden by default,
        # revealed by the Edit button. The master Show toggle moved up into
        # the action row.
        drawer = widgets.VBox(
            [segments_box],
            layout=widgets.Layout(display="none"),
        )
        drawer.add_class("nbpoc-card-drawer")

        # Card title + severity tag. Severity stays 'normal' until the SEG is
        # parsed and we can count segments; updated lazily inside _ensure_parsed.
        title_html = widgets.HTML(
            value=(
                f"<div class='nbpoc-card-head'>"
                f"<div class='nbpoc-card-title'>{path.stem}</div>"
                f"<span class='nbpoc-card-severity normal'>normal</span>"
                f"</div>"
                f"<div class='nbpoc-card-meta'>"
                f"<span>See report</span>"
                f"<span class='sep'>&middot;</span>"
                f"<span>{path.name}</span>"
                f"<span class='sep'>&middot;</span>"
                f"<span>50%</span>"
                f"</div>"
            )
        )

        card = widgets.VBox(
            [title_html, action_row, drawer],
            layout=widgets.Layout(margin="0 0 10px 0"),
        )
        card.add_class("nbpoc-card")
        card.add_class("normal")

        entry["checkbox"] = checkbox
        entry["expand_btn"] = edit_btn
        entry["segments_box"] = segments_box
        entry["accept_btn"] = accept_btn
        entry["reject_btn"] = reject_btn
        entry["title_html"] = title_html
        entry["card"] = card
        entry["drawer"] = drawer

        checkbox.observe(_on_mask_toggle(entry), names="value")
        edit_btn.on_click(_on_expand_clicked(entry))

        def _on_reject(_btn):
            if checkbox.value:
                checkbox.value = False
            card.layout.display = "none"

        def _on_accept(_btn):
            accept_btn.disabled = True

        reject_btn.on_click(_on_reject)
        accept_btn.on_click(_on_accept)

        entry["row"] = card
        return entry

    def _discover_masks():
        nonlocal _masks
        _masks = []
        mask_list_box.children = []
        header_label.value = "<div class='nbpoc-section-label'>RESULTS (0)</div>"

        masks_dir = _masks_dir_for_series()
        if masks_dir is None:
            status_html.value = _sv_muted_card("Load a series to see available masks.")
            return
        if not masks_dir.is_dir():
            status_html.value = _sv_muted_card(
                f"No <code>{_SV_MASKS_SUBDIR}/</code> folder next to this series "
                f"(<code>{masks_dir}</code>). Run a segmentation to generate one."
            )
            return

        dcm_files = sorted(p for p in masks_dir.glob("*.dcm") if p.is_file())
        if not dcm_files:
            status_html.value = _sv_muted_card(
                f"No <code>.dcm</code> files in <code>{masks_dir}</code>."
            )
            return

        rows = []
        for p in dcm_files:
            entry = _build_mask_entry(p)
            _masks.append(entry)
            rows.append(entry["row"])

        mask_list_box.children = rows
        header_label.value = (
            f"<div class='nbpoc-section-label'>RESULTS ({len(dcm_files)})</div>"
        )
        status_html.value = ""

    # --- event handlers -----------------------------------------------------

    def _on_alpha_change(_change):
        _recomposite()

    def _on_rotate(_btn):
        _rotation[0] = (_rotation[0] + 1) % 4
        rotate_btn.button_style = "info" if _rotation[0] else ""
        _recomposite()

    def _on_flip_h(_btn):
        _flip_h[0] = not _flip_h[0]
        flip_h_btn.button_style = "info" if _flip_h[0] else ""
        _recomposite()

    def _on_flip_v(_btn):
        _flip_v[0] = not _flip_v[0]
        flip_v_btn.button_style = "info" if _flip_v[0] else ""
        _recomposite()

    def _on_series_dir_change(_change):
        _reset_caches()
        _discover_masks()
        _restore_originals()

    def _on_series_datasets_change(_change):
        # series_png_cache is repopulated by file_browser when a new series
        # loads; capture a fresh snapshot before any compositing happens.
        _reset_caches()

    def _on_refresh(_btn):
        # Preserve the active overlay set across a rescan by carrying enabled
        # filenames forward — running a fresh segmentation often adds a new
        # mask but should not silently disable ones the user already toggled.
        previously_enabled = {
            entry["name"]
            for entry in _masks
            if entry["checkbox"].value
        }
        _discover_masks()
        if not previously_enabled:
            return
        any_restored = False
        for entry in _masks:
            if entry["name"] in previously_enabled:
                entry["checkbox"].value = True
                any_restored = True
        if not any_restored:
            _recomposite()

    alpha_slider.observe(_on_alpha_change, names="value")
    rotate_btn.on_click(_on_rotate)
    flip_h_btn.on_click(_on_flip_h)
    flip_v_btn.on_click(_on_flip_v)
    refresh_btn.on_click(_on_refresh)
    state.observe(_on_series_dir_change, names="series_dir_path")
    state.observe(_on_series_datasets_change, names="series_datasets")

    _discover_masks()

    diagnostics_panel = build_diagnostics_panel(state)

    return widgets.VBox(
        [
            header,
            status_html,
            mask_list_box,
            display_controls_label,
            alpha_slider,
            orient_row,
            diagnostics_panel,
        ],
        layout=widgets.Layout(width="100%", padding="0"),
    )

# === Inference panel (utils/components/inference_panel.py) =============

"""Right-pane inference panel — disclaimer + status + Analyze + results.

Visually wraps the existing segmentation form (`segmentation_tab`) and the
mask-discovery + display (`seg_viewer`, which itself embeds the diagnostics
panel) inside the clinical-workstation chrome: header + amber disclaimer +
Ready/Stop status row. The big blue "Analyze Current Slice" button is the
existing segmentation run-button restyled at the segmentation_tab layer.

This panel is presentational only — all inference logic stays in the wrapped
modules. Naming choice: "Inference Panel" (not "AI Findings") because the
pipeline today is KServe segmentation, not an LLM/VLM. See memory note
`feedback_inference_vs_ai_naming`.
"""


import ipywidgets as widgets



_INFP_HEADER_HTML = (
    "<div class='nbpoc-inference-header'>"
    "<span class='title'><span class='glyph'>&#x2726;</span> Inference Panel</span>"
    "<span style='font-size:16px;color:var(--text-dim);cursor:pointer;'>&#x2715;</span>"
    "</div>"
)

_INFP_DISCLAIMER_HTML = (
    "<div class='nbpoc-disclaimer'>"
    "Inference Output &mdash; Not a Clinical Diagnosis. "
    "All results must be reviewed by a qualified radiologist."
    "</div>"
)

_INFP_STATUS_HTML = (
    "<div class='nbpoc-status-row'>"
    "<span class='nbpoc-status'><span class='dot'></span> Ready</span>"
    "<button class='nbpoc-stop-btn' type='button'>Stop</button>"
    "</div>"
)


def build_inference_panel(state, viewer) -> widgets.VBox:
    """Build the right-side Inference Panel.

    Composes:
      * static chrome (header, disclaimer, status row)
      * existing `segmentation_tab` (task dropdown + form + Analyze button)
      * existing `seg_viewer` (mask list + transform controls + diagnostics)

    Returns:
        VBox with the `nbpoc-inference` CSS class. Caller embeds this in the
        AppLayout's `right_sidebar` region.
    """

    header = widgets.HTML(value=_INFP_HEADER_HTML)
    disclaimer = widgets.HTML(value=_INFP_DISCLAIMER_HTML)
    status_row = widgets.HTML(value=_INFP_STATUS_HTML)

    segmentation_form = build_segmentation(state)
    segmentation_form.add_class("nbpoc-inference-form")

    seg_viewer_panel = build_seg_viewer(state, viewer)
    seg_viewer_panel.add_class("nbpoc-inference-results")

    body = widgets.VBox(
        [header, disclaimer, status_row, segmentation_form, seg_viewer_panel],
        layout=widgets.Layout(width="100%", padding="0 0 12px 0"),
    )
    body.add_class("nbpoc-inference")
    return body

# === Dashboard assembler (utils/dashboard.py) ==========================

"""Dashboard orchestrator — assembles the modern-gui AppLayout.

Five-region layout filling the viewport:

    +-----------------------------------------------------+
    |                  top toolbar                        |   header
    +--------+-----------------------------+--------------+
    |  scan  |     viewer + overlays       |  inference   |
    |  list  |                             |    panel     |
    +--------+-----------------------------+--------------+
    |                    footer                           |
    +-----------------------------------------------------+

All styling lives in `utils/styles/dark.css`, injected once at app boot via
a <style> HTML widget. No inline CSS in this module.
"""


import warnings
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display


warnings.filterwarnings("ignore", category=UserWarning)



def _load_styles() -> widgets.HTML:
    """Return the embedded dark.css wrapped in a <style> widget."""
    return widgets.HTML(value=f"<style>{_DARK_CSS_TEXT}</style>")


def _dashboard_footer() -> widgets.HTML:
    return widgets.HTML(
        value=(
            "<div class='nbpoc-footer'>"
            "<span>AI-NB-POC &middot; KServe inference &middot; modern-gui</span>"
            "</div>"
        )
    )


def build_and_display_app():
    """Build the modern-gui dashboard and display it."""

    state = AppState()

    css = _load_styles()
    toolbar = build_top_toolbar()
    viewer = build_viewer(state)
    image_browser = build_image_browser(state, viewer)
    inference_panel = build_inference_panel(state, viewer)
    footer = _dashboard_footer()

    layout = widgets.AppLayout(
        header=toolbar,
        left_sidebar=image_browser,
        center=viewer["viewer_panel"],
        right_sidebar=inference_panel,
        footer=footer,
        pane_widths=["280px", 1, "360px"],
        pane_heights=["52px", 1, "28px"],
        merge=False,
    )
    layout.add_class("nbpoc-app")

    container = widgets.VBox(
        [css, layout],
        layout=widgets.Layout(width="100%"),
    )
    display(container)


In [ ]:
import traceback
from IPython.display import display, HTML

try:
    build_and_display_app()
except Exception:
    display(HTML(
        "<pre style='color:#d32f2f;background:#fff3f3;padding:12px;"
        "border-radius:6px;font-size:12px;white-space:pre-wrap;'>"
        f"{traceback.format_exc()}</pre>"
    ))